In [ ]:

# ============================================
# PLC + HMI/SCADA - Silo de Harina para Colab
# Simulación avanzada:
# - 2 silos con conmutación automática
# - Transporte neumático
# - Dosificación a mezcladora/amasadora
# - AUTO / MANUAL
# - Alarmas LL/L/H/HH
# - Paro de emergencia
# - Dashboard HTML interactivo embebido en Colab
# ============================================

from IPython.display import HTML, display
from pathlib import Path

html = r"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>PLC Harina - 2 Silos + Mezcladora</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<style>
  :root{
    --bg:#08111f; --panel:#101b31; --panel2:#172645; --line:rgba(255,255,255,.08);
    --text:#eef4ff; --muted:#9fb1d1; --green:#22c55e; --amber:#f59e0b; --red:#ef4444;
    --blue:#3b82f6; --cyan:#06b6d4; --purple:#8b5cf6; --gray:#64748b;
  }
  *{box-sizing:border-box}
  body{margin:0;font-family:Segoe UI,Arial,sans-serif;background:linear-gradient(180deg,#07101d,#08111f);color:var(--text)}
  .wrap{max-width:1700px;margin:0 auto;padding:20px}
  .header{display:flex;justify-content:space-between;align-items:flex-start;gap:16px;flex-wrap:wrap;margin-bottom:16px}
  h1{margin:0;font-size:30px}
  .sub{color:var(--muted);margin-top:6px}
  .toolbar{display:flex;gap:10px;flex-wrap:wrap}
  button{
    border:none;border-radius:14px;padding:11px 16px;color:#fff;font-weight:700;cursor:pointer;
    box-shadow:0 8px 20px rgba(0,0,0,.22)
  }
  .b-green{background:linear-gradient(135deg,#16a34a,#22c55e)}
  .b-orange{background:linear-gradient(135deg,#ea580c,#f59e0b)}
  .b-blue{background:linear-gradient(135deg,#2563eb,#3b82f6)}
  .b-purple{background:linear-gradient(135deg,#7c3aed,#8b5cf6)}
  .b-red{background:linear-gradient(135deg,#dc2626,#ef4444)}
  .b-gray{background:linear-gradient(135deg,#334155,#475569)}
  .chip{background:var(--panel);border:1px solid var(--line);padding:10px 12px;border-radius:14px;display:flex;align-items:center;gap:10px}
  .chip label{font-size:13px;color:var(--muted)}
  input[type=range]{width:180px}
  .grid-kpi{display:grid;grid-template-columns:repeat(10,1fr);gap:12px;margin-bottom:16px}
  .card{
    background:linear-gradient(180deg,rgba(255,255,255,.02),rgba(255,255,255,.01));
    border:1px solid var(--line);border-radius:18px;padding:16px;box-shadow:0 8px 22px rgba(0,0,0,.22)
  }
  .kpi .label{font-size:12px;color:var(--muted);text-transform:uppercase;letter-spacing:.8px}
  .kpi .value{font-size:25px;font-weight:800;margin-top:8px}
  .kpi .subv{font-size:12px;color:var(--muted);margin-top:8px}
  .green{color:var(--green)} .amber{color:var(--amber)} .red{color:var(--red)} .blue{color:var(--blue)} .purple{color:var(--purple)}
  .main{display:grid;grid-template-columns:1.15fr .85fr;gap:16px;margin-bottom:16px}
  .panel-title{font-size:16px;font-weight:700;margin-bottom:12px}
  .process-box{background:radial-gradient(circle at top left, rgba(59,130,246,.08), transparent 30%), var(--panel2);border:1px solid var(--line);border-radius:18px;padding:14px}
  .right-stack{display:grid;grid-template-rows:auto auto auto;gap:16px}
  .io-grid{display:grid;grid-template-columns:repeat(2,1fr);gap:12px}
  .io{background:rgba(255,255,255,.03);border:1px solid var(--line);border-radius:14px;padding:12px}
  .io .name{font-size:13px;color:var(--muted)}
  .io .state{font-size:21px;font-weight:800;margin:8px 0}
  .tag{font-size:12px;color:var(--muted)}
  .controls{display:grid;grid-template-columns:repeat(2,1fr);gap:10px}
  .controls button{padding:10px 12px;border-radius:12px}
  .alarms{max-height:220px;overflow:auto;background:#07101c;border-radius:14px;padding:10px;border:1px solid var(--line)}
  .alarm-line{padding:7px 4px;border-bottom:1px solid rgba(255,255,255,.05);font-family:Consolas,monospace;font-size:12px}
  .alarm-hh{color:#fca5a5}.alarm-h{color:#fcd34d}.alarm-i{color:#bfdbfe}.alarm-ok{color:#86efac}
  .bottom{display:grid;grid-template-columns:1fr 1fr;gap:16px}
  .table-wrap{overflow:auto;max-height:340px}
  table{width:100%;border-collapse:collapse;font-size:13px}
  th,td{padding:10px 8px;border-bottom:1px solid rgba(255,255,255,.06);text-align:left}
  th{font-size:12px;color:var(--muted);text-transform:uppercase}
  .footer-note{margin-top:10px;color:var(--muted);font-size:12px}
  .lamp{width:12px;height:12px;border-radius:50%;display:inline-block;margin-right:8px;vertical-align:middle}
  .on{background:var(--green);box-shadow:0 0 10px rgba(34,197,94,.75)}
  .off{background:#475569}.al{background:var(--red);box-shadow:0 0 10px rgba(239,68,68,.85)}
  .mini-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-top:12px}
  .mini{background:rgba(255,255,255,.03);border:1px solid var(--line);border-radius:12px;padding:10px}
  .mini .label{font-size:12px;color:var(--muted)} .mini .value{font-weight:800;font-size:20px;margin-top:5px}
  .charts-grid{display:grid;grid-template-columns:1fr 1fr;gap:12px}
  .chart-box{background:#09111f;border-radius:14px;border:1px solid var(--line);padding:10px}
  .chart-box canvas{width:100%;height:240px !important}
  pre{
    margin:0;background:#09111f;border:1px solid var(--line);border-radius:14px;padding:12px;
    color:#dbeafe;font-family:Consolas,monospace;font-size:12px;overflow:auto;max-height:300px
  }
  @media (max-width:1400px){.grid-kpi{grid-template-columns:repeat(5,1fr)}.main,.bottom{grid-template-columns:1fr}}
  @media (max-width:800px){.grid-kpi{grid-template-columns:repeat(2,1fr)}.io-grid,.controls,.charts-grid,.mini-grid{grid-template-columns:1fr}}
</style>
</head>
<body>
<div class="wrap">
  <div class="header">
    <div>
      <h1>PLC - Silo de Harina con 2 Silos + Mezcladora</h1>
      <div class="sub">Conmutación automática · transporte neumático · dosificación · ladder documentado · HMI/SCADA</div>
    </div>
    <div class="toolbar">
      <button class="b-green" onclick="startSim()">Iniciar</button>
      <button class="b-orange" onclick="pauseSim()">Pausar</button>
      <button class="b-blue" onclick="stepSim()">Paso</button>
      <button class="b-purple" onclick="resetSim()">Reset</button>
      <button class="b-red" onclick="emergencyStop()">Paro emergencia</button>
      <div class="chip"><label>Velocidad</label><input id="speedRange" type="range" min="200" max="1800" step="100" value="700"><span id="speedLabel">700 ms</span></div>
    </div>
  </div>

  <div class="grid-kpi">
    <div class="card kpi"><div class="label">Modo</div><div class="value blue" id="kpiModo">AUTO</div><div class="subv">HS-101</div></div>
    <div class="card kpi"><div class="label">Silo activo</div><div class="value purple" id="kpiSiloActivo">S1</div><div class="subv">Conmutación automática</div></div>
    <div class="card kpi"><div class="label">Nivel Silo 1</div><div class="value" id="kpiS1">0%</div><div class="subv">LT-101</div></div>
    <div class="card kpi"><div class="label">Nivel Silo 2</div><div class="value" id="kpiS2">0%</div><div class="subv">LT-201</div></div>
    <div class="card kpi"><div class="label">Tolva mezcladora</div><div class="value" id="kpiMixer">0%</div><div class="subv">LT-301</div></div>
    <div class="card kpi"><div class="label">Demanda proceso</div><div class="value" id="kpiDemanda">0%</div><div class="subv">MW10</div></div>
    <div class="card kpi"><div class="label">Entrada neumática</div><div class="value" id="kpiIn">0.0</div><div class="subv">t/h equiv.</div></div>
    <div class="card kpi"><div class="label">Descarga a mixer</div><div class="value" id="kpiOut">0.0</div><div class="subv">t/h equiv.</div></div>
    <div class="card kpi"><div class="label">Estado proceso</div><div class="value green" id="kpiEstado">Estable</div><div class="subv" id="kpiEstadoSub">Operación normal</div></div>
    <div class="card kpi"><div class="label">Alarmas activas</div><div class="value amber" id="kpiAlarmas">0</div><div class="subv">LL/L/H/HH/E-STOP</div></div>
  </div>

  <div class="main">
    <div class="card">
      <div class="panel-title">Sinótico de proceso</div>
      <div class="process-box">
        <svg id="plantSvg" viewBox="0 0 980 560" width="100%" height="560">
          <text x="30" y="40" fill="#dbeafe" font-size="18" font-weight="700">Recepción / transporte neumático</text>
          <rect id="blower" x="30" y="65" width="120" height="44" rx="12" fill="#475569" stroke="#93c5fd" stroke-width="2"/>
          <text x="50" y="92" fill="white" font-size="14">BL-101</text>
          <rect id="rotary" x="170" y="65" width="120" height="44" rx="12" fill="#475569" stroke="#93c5fd" stroke-width="2"/>
          <text x="184" y="92" fill="white" font-size="14">RV-101</text>
          <line id="pipeMain" x1="290" y1="87" x2="780" y2="87" stroke="#64748b" stroke-width="10" stroke-linecap="round"/>

          <!-- diverter -->
          <rect id="diverter" x="500" y="60" width="80" height="54" rx="12" fill="#334155" stroke="#93c5fd" stroke-width="2"/>
          <text x="511" y="92" fill="white" font-size="13">DV-101</text>
          <line id="branchS1" x1="540" y1="114" x2="540" y2="145" stroke="#64748b" stroke-width="8" />
          <line id="branchS2" x1="720" y1="114" x2="720" y2="145" stroke="#64748b" stroke-width="8" />

          <!-- Silo 1 -->
          <text x="440" y="165" fill="#dbeafe" font-size="18" font-weight="700">SILO 1</text>
          <path d="M390 185 L520 185 L545 330 L365 330 Z" fill="#0f172a" stroke="#cbd5e1" stroke-width="3"/>
          <path d="M365 330 L545 330 L455 425 Z" fill="#0f172a" stroke="#cbd5e1" stroke-width="3"/>
          <polygon id="fillS1" points="390,330 520,330 455,425" fill="#f5f5f4" opacity="0.92"></polygon>
          <rect x="425" y="160" width="60" height="24" rx="8" fill="#1d4ed8" stroke="#93c5fd" stroke-width="2"/><text x="436" y="176" fill="white" font-size="12">LT-101</text>

          <!-- Silo 2 -->
          <text x="620" y="165" fill="#dbeafe" font-size="18" font-weight="700">SILO 2</text>
          <path d="M570 185 L700 185 L725 330 L545 330 Z" fill="#0f172a" stroke="#cbd5e1" stroke-width="3"/>
          <path d="M545 330 L725 330 L635 425 Z" fill="#0f172a" stroke="#cbd5e1" stroke-width="3"/>
          <polygon id="fillS2" points="570,330 700,330 635,425" fill="#f5f5f4" opacity="0.92"></polygon>
          <rect x="605" y="160" width="60" height="24" rx="8" fill="#1d4ed8" stroke="#93c5fd" stroke-width="2"/><text x="616" y="176" fill="white" font-size="12">LT-201</text>

          <!-- outlet screws -->
          <rect id="cvS1" x="378" y="435" width="154" height="24" rx="10" fill="#475569" stroke="#93c5fd" stroke-width="2"/>
          <text x="403" y="451" fill="white" font-size="13">CV-102 S1</text>
          <rect id="cvS2" x="558" y="435" width="154" height="24" rx="10" fill="#475569" stroke="#93c5fd" stroke-width="2"/>
          <text x="583" y="451" fill="white" font-size="13">CV-202 S2</text>

          <!-- mixer -->
          <text x="790" y="220" fill="#dbeafe" font-size="18" font-weight="700">MEZCLADORA</text>
          <rect x="790" y="240" width="130" height="170" rx="18" fill="#0f172a" stroke="#cbd5e1" stroke-width="3"/>
          <rect id="fillMixer" x="805" y="370" width="100" height="25" rx="10" fill="#f5f5f4" opacity="0.92"/>
          <rect x="820" y="214" width="70" height="24" rx="8" fill="#1d4ed8" stroke="#93c5fd" stroke-width="2"/><text x="831" y="230" fill="white" font-size="12">LT-301</text>
          <rect id="agit" x="835" y="417" width="40" height="40" rx="8" fill="#475569" stroke="#93c5fd" stroke-width="2"/>
          <text x="828" y="444" fill="white" font-size="12">AG-301</text>

          <!-- pipes to mixer -->
          <line id="lineS1ToMix" x1="530" y1="447" x2="790" y2="447" stroke="#64748b" stroke-width="8" stroke-linecap="round"/>
          <line id="lineS2ToMix" x1="710" y1="447" x2="790" y2="447" stroke="#64748b" stroke-width="8" stroke-linecap="round"/>

          <!-- animated flow -->
          <circle id="p1" cx="320" cy="87" r="7" fill="#38bdf8" opacity="0"/>
          <circle id="p2" cx="360" cy="87" r="7" fill="#38bdf8" opacity="0"/>
          <circle id="p3" cx="400" cy="87" r="7" fill="#38bdf8" opacity="0"/>
          <circle id="o1" cx="560" cy="447" r="7" fill="#22c55e" opacity="0"/>
          <circle id="o2" cx="610" cy="447" r="7" fill="#22c55e" opacity="0"/>
          <circle id="o3" cx="660" cy="447" r="7" fill="#22c55e" opacity="0"/>

          <text x="30" y="530" fill="#94a3b8" font-size="12">
            Estrategia: llenar silo activo; si su nivel cae bajo LL se conmuta al otro silo disponible. Dosificación automática a mezcladora según nivel y demanda.
          </text>
        </svg>

        <div class="mini-grid">
          <div class="mini"><div class="label">LSLL S1</div><div class="value" id="mS1LL">0</div></div>
          <div class="mini"><div class="label">LSHH S1</div><div class="value" id="mS1HH">0</div></div>
          <div class="mini"><div class="label">LSLL S2</div><div class="value" id="mS2LL">0</div></div>
          <div class="mini"><div class="label">LSHH S2</div><div class="value" id="mS2HH">0</div></div>
        </div>
      </div>
    </div>

    <div class="right-stack">
      <div class="card">
        <div class="panel-title">Entradas / salidas PLC</div>
        <div class="io-grid">
          <div class="io"><div class="name">Modo</div><div class="state" id="ioMode">AUTO</div><div class="tag">HS-101</div></div>
          <div class="io"><div class="name">Paro emergencia</div><div class="state" id="ioEStop">OFF</div><div class="tag">ES-101</div></div>
          <div class="io"><div class="name">Soplador neumático</div><div class="state" id="ioBlower"><span class="lamp off"></span>OFF</div><div class="tag">Q0.0 / BL-101</div></div>
          <div class="io"><div class="name">Válvula rotativa</div><div class="state" id="ioRotary"><span class="lamp off"></span>OFF</div><div class="tag">Q0.1 / RV-101</div></div>
          <div class="io"><div class="name">Desviadora S1</div><div class="state" id="ioDivS1"><span class="lamp off"></span>OFF</div><div class="tag">Q0.2 / DV-S1</div></div>
          <div class="io"><div class="name">Desviadora S2</div><div class="state" id="ioDivS2"><span class="lamp off"></span>OFF</div><div class="tag">Q0.3 / DV-S2</div></div>
          <div class="io"><div class="name">Tornillo salida S1</div><div class="state" id="ioCv1"><span class="lamp off"></span>OFF</div><div class="tag">Q0.4 / CV-102</div></div>
          <div class="io"><div class="name">Tornillo salida S2</div><div class="state" id="ioCv2"><span class="lamp off"></span>OFF</div><div class="tag">Q0.5 / CV-202</div></div>
          <div class="io"><div class="name">Agitador mixer</div><div class="state" id="ioAgit"><span class="lamp off"></span>OFF</div><div class="tag">Q0.6 / AG-301</div></div>
          <div class="io"><div class="name">Nivel mixer</div><div class="state" id="ioMix">0%</div><div class="tag">IW90 / LT-301</div></div>
        </div>
      </div>

      <div class="card">
        <div class="panel-title">Controles HMI</div>
        <div class="controls">
          <button class="b-blue" onclick="toggleMode()">Cambiar AUTO/MAN</button>
          <button class="b-gray" onclick="ackAlarms()">Reconocer alarmas</button>
          <button class="b-green" onclick="manualToggle('blower')">Manual BL-101</button>
          <button class="b-green" onclick="manualToggle('rotary')">Manual RV-101</button>
          <button class="b-green" onclick="manualToggle('divS1')">Manual DV→S1</button>
          <button class="b-green" onclick="manualToggle('divS2')">Manual DV→S2</button>
          <button class="b-green" onclick="manualToggle('cv1')">Manual CV-102</button>
          <button class="b-green" onclick="manualToggle('cv2')">Manual CV-202</button>
          <button class="b-green" onclick="manualToggle('agit')">Manual AG-301</button>
          <button class="b-purple" onclick="setDemand('low')">Demanda baja</button>
          <button class="b-purple" onclick="setDemand('med')">Demanda media</button>
          <button class="b-purple" onclick="setDemand('high')">Demanda alta</button>
          <button class="b-red" onclick="forceAlarm('s1')">Prueba HH S1</button>
          <button class="b-red" onclick="forceAlarm('s2')">Prueba HH S2</button>
        </div>
        <div class="footer-note">En automático el PLC decide llenado, conmutación de silo fuente y dosificación a mezcladora.</div>
      </div>

      <div class="card">
        <div class="panel-title">Alarmas y eventos</div>
        <div class="alarms" id="alarmBox"></div>
      </div>
    </div>
  </div>

  <div class="bottom">
    <div class="card">
      <div class="panel-title">Tendencias</div>
      <div class="charts-grid">
        <div class="chart-box"><canvas id="chartLevel"></canvas></div>
        <div class="chart-box"><canvas id="chartMixer"></canvas></div>
      </div>
      <div class="footer-note">Tendencias históricas de ambos silos y tolva de mezcladora.</div>
    </div>

    <div class="card">
      <div class="panel-title">Tabla de tags PLC</div>
      <div class="table-wrap">
        <table>
          <thead><tr><th>Tag</th><th>Dirección</th><th>Descripción</th><th>Valor</th></tr></thead>
          <tbody id="tagTable"></tbody>
        </table>
      </div>
    </div>
  </div>

  <div style="height:16px"></div>
  <div class="bottom">
    <div class="card">
      <div class="panel-title">Lógica Ladder documentada</div>
<pre>
RUNG 1 - MODO AUTOMÁTICO
|----[ HS-101 AUTO ]---------------------------------------------( M_AUTO )----|

RUNG 2 - PARO DE EMERGENCIA
|----[ ES-101 ]--------------------------------------------------( ALM_ESTOP )-|
|----[ /ES-101 ]-------------------------------------------------( PERMISO )---|

RUNG 3 - LLENADO SILO ACTIVO
|----[ M_AUTO ]--[ PERMISO ]--[ NIVEL_SILO_ACTIVO < 75 ]---------( BL-101 )----|
|----[ M_AUTO ]--[ PERMISO ]--[ NIVEL_SILO_ACTIVO < 75 ]---------( RV-101 )----|

RUNG 4 - DESVIADORA DE LÍNEA
|----[ SILO_ACTIVO = S1 ]----------------------------------------( DV_S1 )------|
|----[ SILO_ACTIVO = S2 ]----------------------------------------( DV_S2 )------|

RUNG 5 - CONMUTACIÓN AUTOMÁTICA
|----[ SILO_ACTIVO = S1 ]--[ LT-101 <= 10 ]--[ LT-201 > 10 ]-----( CAMBIAR_S2 )|
|----[ SILO_ACTIVO = S2 ]--[ LT-201 <= 10 ]--[ LT-101 > 10 ]-----( CAMBIAR_S1 )|

RUNG 6 - DOSIFICACIÓN A MEZCLADORA
|----[ M_AUTO ]--[ LT-301 < 65 ]--[ SILO_ACTIVO = S1 ]----------( CV-102 )-----|
|----[ M_AUTO ]--[ LT-301 < 65 ]--[ SILO_ACTIVO = S2 ]----------( CV-202 )-----|
|----[ LT-301 < 80 ]--------------------------------------------( AG-301 )------|

RUNG 7 - ALARMAS
|----[ LT-101 >= 90 ]-------------------------------------------( ALM_HH_S1 )---|
|----[ LT-101 <= 10 ]-------------------------------------------( ALM_LL_S1 )---|
|----[ LT-201 >= 90 ]-------------------------------------------( ALM_HH_S2 )---|
|----[ LT-201 <= 10 ]-------------------------------------------( ALM_LL_S2 )---|
</pre>
    </div>

    <div class="card">
      <div class="panel-title">Secuencia funcional</div>
<pre>
1) El operador selecciona AUTO.
2) Si no hay emergencia, el PLC habilita transporte neumático.
3) La desviadora dirige el producto al silo activo.
4) El silo activo se llena hasta banda alta.
5) La mezcladora consume harina según la demanda del proceso.
6) Cuando la tolva de mezcladora baja, el PLC abre el tornillo del silo activo.
7) Si el silo activo cae a nivel muy bajo, conmuta al otro silo disponible.
8) Si ambos silos quedan bajos, mantiene alarmas y limita dosificación.
9) Si un silo llega a HH, detiene llenado de ese silo.
10) Con E-STOP, todas las salidas van a OFF.
</pre>
    </div>
  </div>
</div>

<script>
const CFG = {
  tickMs: 700,
  histMax: 80,
  lowLow:10, low:25, high:75, highHigh:90,
  inletBase:1.15,
  outletBase:0.70,
  mixerConsumeBase:0.55,
  noise:0.08
};

let timer = null, scan = 0;
let hist = {labels:[], s1:[], s2:[], mix:[]};
let alarmLog = [];

const plc = {
  auto:true, eStop:false, active:'S1', demand:55,
  s1:55, s2:82, mixer:44,
  out:{
    blower:false, rotary:false, divS1:true, divS2:false, cv1:false, cv2:false, agit:false
  },
  man:{blower:false, rotary:false, divS1:false, divS2:false, cv1:false, cv2:false, agit:false},
  al:{s1LL:false,s1L:false,s1H:false,s1HH:false,s2LL:false,s2L:false,s2H:false,s2HH:false, estop:false}
};

function clamp(v,min,max){ return Math.max(min, Math.min(max,v)); }
function rnd(a,b){ return Math.random()*(b-a)+a; }
function ts(){ return new Date().toLocaleTimeString(); }
function activeAlarmCount(){ return Object.values(plc.al).filter(Boolean).length; }

function pushEvent(cls,msg){
  alarmLog.unshift({cls,msg,time:ts()});
  alarmLog = alarmLog.slice(0,140);
  document.getElementById('alarmBox').innerHTML = alarmLog.map(a => `<div class="alarm-line alarm-${a.cls}">[${a.time}] ${a.msg}</div>`).join('');
}

function lamp(on, alarm=false){
  if(alarm) return '<span class="lamp al"></span>ON';
  return `<span class="lamp ${on?'on':'off'}"></span>${on?'ON':'OFF'}`;
}

function updateDiscrete(){
  const prev = {...plc.al};
  plc.al.s1LL = plc.s1 <= CFG.lowLow; plc.al.s1L = plc.s1 <= CFG.low; plc.al.s1H = plc.s1 >= CFG.high; plc.al.s1HH = plc.s1 >= CFG.highHigh;
  plc.al.s2LL = plc.s2 <= CFG.lowLow; plc.al.s2L = plc.s2 <= CFG.low; plc.al.s2H = plc.s2 >= CFG.high; plc.al.s2HH = plc.s2 >= CFG.highHigh;
  plc.al.estop = plc.eStop;

  if(plc.al.s1LL && !prev.s1LL) pushEvent('hh','ALM-LL S1: nivel muy bajo');
  if(plc.al.s1HH && !prev.s1HH) pushEvent('hh','ALM-HH S1: nivel muy alto');
  if(plc.al.s2LL && !prev.s2LL) pushEvent('hh','ALM-LL S2: nivel muy bajo');
  if(plc.al.s2HH && !prev.s2HH) pushEvent('hh','ALM-HH S2: nivel muy alto');
  if(plc.al.estop && !prev.estop) pushEvent('hh','ALM-E-STOP: emergencia activa');
}

function plcLogic(){
  if(plc.eStop){
    Object.keys(plc.out).forEach(k => plc.out[k] = false);
    return;
  }

  if(plc.auto){
    // conmutación automática
    if(plc.active === 'S1' && plc.s1 <= CFG.lowLow && plc.s2 > CFG.lowLow){
      plc.active = 'S2';
      pushEvent('ok','Conmutación automática: Silo activo = S2');
    } else if(plc.active === 'S2' && plc.s2 <= CFG.lowLow && plc.s1 > CFG.lowLow){
      plc.active = 'S1';
      pushEvent('ok','Conmutación automática: Silo activo = S1');
    }

    // transporte neumático hacia silo activo si no está alto
    const activeLvl = plc.active === 'S1' ? plc.s1 : plc.s2;
    const canFill = activeLvl < CFG.high && !((plc.active === 'S1' && plc.al.s1HH) || (plc.active === 'S2' && plc.al.s2HH));
    plc.out.blower = canFill;
    plc.out.rotary = canFill;
    plc.out.divS1 = plc.active === 'S1';
    plc.out.divS2 = plc.active === 'S2';

    // dosificación a mezcladora
    plc.out.cv1 = false; plc.out.cv2 = false;
    if(plc.mixer < 65){
      if(plc.active === 'S1' && plc.s1 > CFG.lowLow + 1) plc.out.cv1 = true;
      if(plc.active === 'S2' && plc.s2 > CFG.lowLow + 1) plc.out.cv2 = true;
    }
    plc.out.agit = plc.mixer < 80;
  } else {
    for (const k of Object.keys(plc.out)) plc.out[k] = plc.man[k];
  }
}

function processModel(){
  let inletRate = 0, outRate = 0;
  const d = plc.demand / 100;

  if(plc.out.blower && plc.out.rotary){
    inletRate = CFG.inletBase + rnd(-CFG.noise, CFG.noise);
    if(plc.out.divS1) plc.s1 = clamp(plc.s1 + inletRate, 0, 100);
    if(plc.out.divS2) plc.s2 = clamp(plc.s2 + inletRate, 0, 100);
  }

  const doseRate = (CFG.outletBase + 1.0*d) + rnd(-CFG.noise, CFG.noise);
  if(plc.out.cv1){
    outRate = doseRate;
    plc.s1 = clamp(plc.s1 - outRate, 0, 100);
    plc.mixer = clamp(plc.mixer + outRate*0.9, 0, 100);
  }
  if(plc.out.cv2){
    outRate = doseRate;
    plc.s2 = clamp(plc.s2 - outRate, 0, 100);
    plc.mixer = clamp(plc.mixer + outRate*0.9, 0, 100);
  }

  // consumo del mixer por amasado
  const mixerUse = (CFG.mixerConsumeBase + 1.1*d) + rnd(-CFG.noise, CFG.noise);
  plc.mixer = clamp(plc.mixer - mixerUse, 0, 100);

  return {inletRate:Math.max(0,inletRate), outRate:Math.max(0,outRate)};
}

function demandProfile(){
  const base = 52 + 28*Math.sin(scan/10);
  const pert = rnd(-8, 8);
  plc.demand = clamp(base + pert, 10, 100);
}

function health(){
  if(plc.eStop) return {txt:'Paro', cls:'red', sub:'Emergencia activa'};
  if(plc.al.s1HH || plc.al.s1LL || plc.al.s2HH || plc.al.s2LL) return {txt:'Alarma', cls:'red', sub:'Revisar niveles'};
  if(plc.out.blower) return {txt:'Llenando', cls:'green', sub:'Transporte neumático activo'};
  if(plc.out.cv1 || plc.out.cv2) return {txt:'Dosificando', cls:'blue', sub:'Alimentando mezcladora'};
  return {txt:'Estable', cls:'green', sub:'Operación normal'};
}

function updateSVG(){
  function setSilo(polyId, level, xLeft, xRight, xMid){
    let points;
    if(level <= 34.5){
      const k = (level / 34.5);
      const leftX = xMid - 90*k, rightX = xMid + 90*k;
      points = `${leftX},330 ${rightX},330 ${xMid},425`;
      if(k > .98) points = `${xLeft},330 ${xRight},330 ${xMid},425`;
    } else {
      const extra = Math.min(145, (level-34.5)*2.9);
      const yTop = 330 - extra;
      points = `${xLeft},${yTop} ${xRight},${yTop} ${xRight},330 ${xMid+90},330 ${xMid},425 ${xMid-90},330 ${xLeft},330`;
    }
    document.getElementById(polyId).setAttribute('points', points);
  }
  setSilo('fillS1', plc.s1, 390, 520, 455);
  setSilo('fillS2', plc.s2, 570, 700, 635);

  const mixH = 140 * (plc.mixer/100);
  document.getElementById('fillMixer').setAttribute('y', 395 - mixH);
  document.getElementById('fillMixer').setAttribute('height', Math.max(12,mixH));

  document.getElementById('blower').setAttribute('fill', plc.out.blower ? '#22c55e' : '#475569');
  document.getElementById('rotary').setAttribute('fill', plc.out.rotary ? '#22c55e' : '#475569');
  document.getElementById('diverter').setAttribute('fill', (plc.out.divS1||plc.out.divS2) ? '#22c55e' : '#334155');
  document.getElementById('branchS1').setAttribute('stroke', plc.out.divS1 ? '#38bdf8' : '#64748b');
  document.getElementById('branchS2').setAttribute('stroke', plc.out.divS2 ? '#38bdf8' : '#64748b');
  document.getElementById('cvS1').setAttribute('fill', plc.out.cv1 ? '#22c55e' : '#475569');
  document.getElementById('cvS2').setAttribute('fill', plc.out.cv2 ? '#22c55e' : '#475569');
  document.getElementById('agit').setAttribute('fill', plc.out.agit ? '#22c55e' : '#475569');
  document.getElementById('lineS1ToMix').setAttribute('stroke', plc.out.cv1 ? '#22c55e' : '#64748b');
  document.getElementById('lineS2ToMix').setAttribute('stroke', plc.out.cv2 ? '#22c55e' : '#64748b');
  document.getElementById('pipeMain').setAttribute('stroke', plc.out.blower ? '#38bdf8' : '#64748b');

  ['p1','p2','p3'].forEach((id,i)=>{
    const el = document.getElementById(id);
    el.setAttribute('opacity', plc.out.blower ? '1' : '0');
    el.setAttribute('cx', 320 + ((scan+i*2)%12)*25);
  });
  ['o1','o2','o3'].forEach((id,i)=>{
    const el = document.getElementById(id);
    const active = plc.out.cv1 || plc.out.cv2;
    el.setAttribute('opacity', active ? '1' : '0');
    el.setAttribute('cx', 560 + ((scan+i*2)%10)*25);
  });
}

function updateKPIs(r){
  document.getElementById('kpiModo').textContent = plc.auto ? 'AUTO' : 'MAN';
  document.getElementById('kpiModo').className = 'value ' + (plc.auto ? 'blue' : 'amber');
  document.getElementById('kpiSiloActivo').textContent = plc.active;
  document.getElementById('kpiS1').textContent = plc.s1.toFixed(1)+'%';
  document.getElementById('kpiS2').textContent = plc.s2.toFixed(1)+'%';
  document.getElementById('kpiMixer').textContent = plc.mixer.toFixed(1)+'%';
  document.getElementById('kpiDemanda').textContent = plc.demand.toFixed(0)+'%';
  document.getElementById('kpiIn').textContent = r.inletRate.toFixed(1);
  document.getElementById('kpiOut').textContent = r.outRate.toFixed(1);
  document.getElementById('kpiAlarmas').textContent = activeAlarmCount();

  const h = health();
  document.getElementById('kpiEstado').textContent = h.txt;
  document.getElementById('kpiEstado').className = 'value ' + h.cls;
  document.getElementById('kpiEstadoSub').textContent = h.sub;

  document.getElementById('ioMode').textContent = plc.auto ? 'AUTO' : 'MANUAL';
  document.getElementById('ioEStop').innerHTML = plc.eStop ? '<span class="lamp al"></span>ON' : '<span class="lamp off"></span>OFF';
  document.getElementById('ioBlower').innerHTML = lamp(plc.out.blower);
  document.getElementById('ioRotary').innerHTML = lamp(plc.out.rotary);
  document.getElementById('ioDivS1').innerHTML = lamp(plc.out.divS1);
  document.getElementById('ioDivS2').innerHTML = lamp(plc.out.divS2);
  document.getElementById('ioCv1').innerHTML = lamp(plc.out.cv1);
  document.getElementById('ioCv2').innerHTML = lamp(plc.out.cv2);
  document.getElementById('ioAgit').innerHTML = lamp(plc.out.agit);
  document.getElementById('ioMix').textContent = plc.mixer.toFixed(1)+'%';

  document.getElementById('mS1LL').textContent = plc.al.s1LL ? '1' : '0';
  document.getElementById('mS1HH').textContent = plc.al.s1HH ? '1' : '0';
  document.getElementById('mS2LL').textContent = plc.al.s2LL ? '1' : '0';
  document.getElementById('mS2HH').textContent = plc.al.s2HH ? '1' : '0';

  updateSVG();
  renderTags(r);
}

function renderTags(r){
  const rows = [
    ['LT-101','IW64','Nivel silo 1', plc.s1.toFixed(1)+' %'],
    ['LT-201','IW74','Nivel silo 2', plc.s2.toFixed(1)+' %'],
    ['LT-301','IW90','Nivel mezcladora', plc.mixer.toFixed(1)+' %'],
    ['HS-101','I0.0','Modo automático', plc.auto ? '1':'0'],
    ['ES-101','I0.1','Paro emergencia', plc.eStop ? '1':'0'],
    ['BL-101','Q0.0','Soplador neumático', plc.out.blower ? '1':'0'],
    ['RV-101','Q0.1','Válvula rotativa', plc.out.rotary ? '1':'0'],
    ['DV-S1','Q0.2','Desviadora a silo 1', plc.out.divS1 ? '1':'0'],
    ['DV-S2','Q0.3','Desviadora a silo 2', plc.out.divS2 ? '1':'0'],
    ['CV-102','Q0.4','Tornillo salida silo 1', plc.out.cv1 ? '1':'0'],
    ['CV-202','Q0.5','Tornillo salida silo 2', plc.out.cv2 ? '1':'0'],
    ['AG-301','Q0.6','Agitador mezcladora', plc.out.agit ? '1':'0'],
    ['MW10','MW10','Demanda proceso', plc.demand.toFixed(0)+' %'],
    ['MW20','MW20','Caudal entrada', r.inletRate.toFixed(2)],
    ['MW21','MW21','Caudal dosificación', r.outRate.toFixed(2)],
    ['MW30','MW30','Silo activo', plc.active],
    ['MW31','MW31','Scan PLC', scan]
  ];
  document.getElementById('tagTable').innerHTML = rows.map(t => `<tr><td>${t[0]}</td><td>${t[1]}</td><td>${t[2]}</td><td>${t[3]}</td></tr>`).join('');
}

let chartLevel = null, chartMixer = null;
function initCharts(){
  const commonOpts = {
    responsive:true, maintainAspectRatio:false,
    plugins:{legend:{labels:{color:'#eef4ff'}}},
    scales:{
      x:{ticks:{color:'#cbd5e1'},grid:{color:'rgba(255,255,255,.06)'}},
      y:{min:0,max:100,ticks:{color:'#cbd5e1'},grid:{color:'rgba(255,255,255,.06)'}}
    }
  };
  chartLevel = new Chart(document.getElementById('chartLevel'), {
    type:'line',
    data:{labels:[], datasets:[
      {label:'Silo 1', data:[], borderColor:'#38bdf8', backgroundColor:'rgba(56,189,248,.15)', fill:false, tension:.3},
      {label:'Silo 2', data:[], borderColor:'#8b5cf6', backgroundColor:'rgba(139,92,246,.15)', fill:false, tension:.3}
    ]},
    options: commonOpts
  });
  chartMixer = new Chart(document.getElementById('chartMixer'), {
    type:'line',
    data:{labels:[], datasets:[
      {label:'Mezcladora', data:[], borderColor:'#22c55e', backgroundColor:'rgba(34,197,94,.15)', fill:true, tension:.3}
    ]},
    options: commonOpts
  });
}

function refreshCharts(){
  chartLevel.data.labels = hist.labels;
  chartLevel.data.datasets[0].data = hist.s1;
  chartLevel.data.datasets[1].data = hist.s2;
  chartLevel.update();

  chartMixer.data.labels = hist.labels;
  chartMixer.data.datasets[0].data = hist.mix;
  chartMixer.update();
}

function tick(){
  scan++;
  if(plc.auto && !plc.eStop) demandProfile();
  plcLogic();
  const r = processModel();
  updateDiscrete();

  hist.labels.push(scan);
  hist.s1.push(Number(plc.s1.toFixed(1)));
  hist.s2.push(Number(plc.s2.toFixed(1)));
  hist.mix.push(Number(plc.mixer.toFixed(1)));
  if(hist.labels.length > CFG.histMax){
    hist.labels.shift(); hist.s1.shift(); hist.s2.shift(); hist.mix.shift();
  }

  updateKPIs(r);
  refreshCharts();
}

function startSim(){ if(timer) return; pushEvent('i','Simulación iniciada'); timer = setInterval(tick, CFG.tickMs); }
function pauseSim(){ if(timer){ clearInterval(timer); timer = null; pushEvent('i','Simulación pausada'); } }
function stepSim(){ pauseSim(); tick(); pushEvent('i','Paso manual ejecutado'); }
function emergencyStop(){
  plc.eStop = !plc.eStop;
  pushEvent(plc.eStop ? 'hh' : 'ok', plc.eStop ? 'Paro de emergencia activado' : 'Paro de emergencia liberado');
  tick();
}
function toggleMode(){
  plc.auto = !plc.auto;
  pushEvent('i','Modo cambiado a ' + (plc.auto ? 'AUTO' : 'MANUAL'));
  updateKPIs({inletRate:0,outRate:0});
}
function manualToggle(key){
  if(plc.auto){ pushEvent('h','Ignorado: pasar a MANUAL'); return; }
  plc.man[key] = !plc.man[key];
  pushEvent('i',`Salida manual ${key} = ${plc.man[key] ? 'ON':'OFF'}`);
  tick();
}
function ackAlarms(){ pushEvent('ok','Alarmas reconocidas por operador'); }
function setDemand(kind){
  const m = {low:20, med:55, high:90};
  plc.demand = m[kind];
  pushEvent('i','Demanda fijada en ' + plc.demand + '%');
  updateKPIs({inletRate:0,outRate:0});
}
function forceAlarm(silo){
  if(silo === 's1') plc.s1 = 93;
  if(silo === 's2') plc.s2 = 93;
  pushEvent('hh','Prueba HH forzada en ' + silo.toUpperCase());
  tick();
}
function resetSim(){
  pauseSim();
  scan = 0;
  hist = {labels:[], s1:[], s2:[], mix:[]};
  alarmLog = [];
  plc.auto = true; plc.eStop = false; plc.active = 'S1'; plc.demand = 55;
  plc.s1 = 55; plc.s2 = 82; plc.mixer = 44;
  plc.out = {blower:false, rotary:false, divS1:true, divS2:false, cv1:false, cv2:false, agit:false};
  plc.man = {blower:false, rotary:false, divS1:false, divS2:false, cv1:false, cv2:false, agit:false};
  plc.al = {s1LL:false,s1L:false,s1H:false,s1HH:false,s2LL:false,s2L:false,s2H:false,s2HH:false, estop:false};
  document.getElementById('alarmBox').innerHTML = '';
  pushEvent('ok','Sistema reiniciado');
  updateKPIs({inletRate:0,outRate:0});
  refreshCharts();
}

document.getElementById('speedRange').addEventListener('input', e => {
  CFG.tickMs = Number(e.target.value);
  document.getElementById('speedLabel').textContent = CFG.tickMs + ' ms';
  if(timer){ clearInterval(timer); timer = setInterval(tick, CFG.tickMs); }
});

initCharts();
resetSim();
</script>
</body>
</html>
"""

out_dir = Path("/mnt/data")
html_path = out_dir / "plc_silos_harina_colab_dashboard.html"
html_path.write_text(html, encoding="utf-8")

iframe = f"""
<div style="width:100%;height:920px;border:1px solid #ccc;border-radius:12px;overflow:hidden;">
  <iframe src="files/{html_path.name}" width="100%" height="100%" style="border:none;"></iframe>
</div>
"""

display(HTML(iframe))
print(f"HTML generado: {html_path}")


In [3]:
%run /content/plc_silos_harina_colab.py

Exception: File `'/content/plc_silos_harina_colab.py'` not found.

The following Python code extracts the JavaScript simulation logic from the HTML string for inspection or further processing.

In [5]:
import re

# Include the definition of 'html' to make this cell self-contained
html = r"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>PLC Harina - 2 Silos + Mezcladora</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<style>
  :root{
    --bg:#08111f; --panel:#101b31; --panel2:#172645; --line:rgba(255,255,255,.08);
    --text:#eef4ff; --muted:#9fb1d1; --green:#22c55e; --amber:#f59e0b; --red:#ef4444;
    --blue:#3b82f6; --cyan:#06b6d4; --purple:#8b5cf6; --gray:#64748b;
  }
  *{
    box-sizing:border-box
  }
  body{
    margin:0;
    font-family:Segoe UI,Arial,sans-serif;
    background:linear-gradient(180deg,#07101d,#08111f);
    color:var(--text)
  }
  .wrap{
    max-width:1700px;
    margin:0 auto;
    padding:20px
  }
  .header{
    display:flex;
    justify-content:space-between;
    align-items:flex-start;
    gap:16px;
    flex-wrap:wrap;
    margin-bottom:16px
  }
  h1{
    margin:0;
    font-size:30px
  }
  .sub{
    color:var(--muted);
    margin-top:6px
  }
  .toolbar{
    display:flex;
    gap:10px;
    flex-wrap:wrap
  }
  button{
    border:none;
    border-radius:14px;
    padding:11px 16px;
    color:#fff;
    font-weight:700;
    cursor:pointer;
    box-shadow:0 8px 20px rgba(0,0,0,.22)
  }
  .b-green{
    background:linear-gradient(135deg,#16a34a,#22c55e)
  }
  .b-orange{
    background:linear-gradient(135deg,#ea580c,#f59e0b)
  }
  .b-blue{
    background:linear-gradient(135deg,#2563eb,#3b82f6)
  }
  .b-purple{
    background:linear-gradient(135deg,#7c3aed,#8b5cf6)
  }
  .b-red{
    background:linear-gradient(135deg,#dc2626,#ef4444)
  }
  .b-gray{
    background:linear-gradient(135deg,#334155,#475569)
  }
  .chip{
    background:var(--panel);
    border:1px solid var(--line);
    padding:10px 12px;
    border-radius:14px;
    display:flex;
    align-items:center;
    gap:10px
  }
  .chip label{
    font-size:13px;
    color:var(--muted)
  }
  input[type=range]{
    width:180px
  }
  .grid-kpi{
    display:grid;
    grid-template-columns:repeat(10,1fr);
    gap:12px;
    margin-bottom:16px
  }
  .card{
    background:linear-gradient(180deg,rgba(255,255,255,.02),rgba(255,255,255,.01));
    border:1px solid var(--line);
    border-radius:18px;
    padding:16px;
    box-shadow:0 8px 22px rgba(0,0,0,.22)
  }
  .kpi .label{
    font-size:12px;
    color:var(--muted);
    text-transform:uppercase;
    letter-spacing:.8px
  }
  .kpi .value{
    font-size:25px;
    font-weight:800;
    margin-top:8px
  }
  .kpi .subv{
    font-size:12px;
    color:var(--muted);
    margin-top:8px
  }
  .green{
    color:var(--green)
  }
  .amber{
    color:var(--amber)
  }
  .red{
    color:var(--red)
  }
  .blue{
    color:var(--blue)
  }
  .purple{
    color:var(--purple)
  }
  .main{
    display:grid;
    grid-template-columns:1.15fr .85fr;
    gap:16px;
    margin-bottom:16px
  }
  .panel-title{
    font-size:16px;
    font-weight:700;
    margin-bottom:12px
  }
  .process-box{
    background:radial-gradient(circle at top left, rgba(59,130,246,.08), transparent 30%), var(--panel2);
    border:1px solid var(--line);
    border-radius:18px;
    padding:14px
  }
  .right-stack{
    display:grid;
    grid-template-rows:auto auto auto;
    gap:16px
  }
  .io-grid{
    display:grid;
    grid-template-columns:repeat(2,1fr);
    gap:12px
  }
  .io{
    background:rgba(255,255,255,.03);
    border:1px solid var(--line);
    border-radius:14px;
    padding:12px
  }
  .io .name{
    font-size:13px;
    color:var(--muted)
  }
  .io .state{
    font-size:21px;
    font-weight:800;
    margin:8px 0
  }
  .tag{
    font-size:12px;
    color:var(--muted)
  }
  .controls{
    display:grid;
    grid-template-columns:repeat(2,1fr);
    gap:10px
  }
  .controls button{
    padding:10px 12px;
    border-radius:12px
  }
  .alarms{
    max-height:220px;
    overflow:auto;
    background:#07101c;
    border-radius:14px;
    padding:10px;
    border:1px solid var(--line)
  }
  .alarm-line{
    padding:7px 4px;
    border-bottom:1px solid rgba(255,255,255,.05);
    font-family:Consolas,monospace;
    font-size:12px
  }
  .alarm-hh{
    color:#fca5a5
  }
  .alarm-h{
    color:#fcd34d
  }
  .alarm-i{
    color:#bfdbfe
  }
  .alarm-ok{
    color:#86efac
  }
  .bottom{
    display:grid;
    grid-template-columns:1fr 1fr;
    gap:16px
  }
  .table-wrap{
    overflow:auto;
    max-height:340px
  }
  table{
    width:100%;
    border-collapse:collapse;
    font-size:13px
  }
  th,td{
    padding:10px 8px;
    border-bottom:1px solid rgba(255,255,255,.06);
    text-align:left
  }
  th{
    font-size:12px;
    color:var(--muted);
    text-transform:uppercase
  }
  .footer-note{
    margin-top:10px;
    color:var(--muted);
    font-size:12px
  }
  .lamp{
    width:12px;
    height:12px;
    border-radius:50%;
    display:inline-block;
    margin-right:8px;
    vertical-align:middle
  }
  .on{
    background:var(--green);
    box-shadow:0 0 10px rgba(34,197,94,.75)
  }
  .off{
    background:#475569
  }
  .al{
    background:var(--red);
    box-shadow:0 0 10px rgba(239,68,68,.85)
  }
  .mini-grid{
    display:grid;
    grid-template-columns:repeat(4,1fr);
    gap:10px;
    margin-top:12px
  }
  .mini{
    background:rgba(255,255,255,.03);
    border:1px solid var(--line);
    border-radius:12px;
    padding:10px
  }
  .mini .label{
    font-size:12px;
    color:var(--muted)
  }
  .mini .value{
    font-weight:800;
    font-size:20px;
    margin-top:5px
  }
  .charts-grid{
    display:grid;
    grid-template-columns:1fr 1fr;
    gap:12px
  }
  .chart-box{
    background:#09111f;
    border-radius:14px;
    border:1px solid var(--line);
    padding:10px
  }
  .chart-box canvas{
    width:100%;
    height:240px !important
  }
  pre{
    margin:0;
    background:#09111f;
    border:1px solid var(--line);
    border-radius:14px;
    padding:12px;
    color:#dbeafe;
    font-family:Consolas,monospace;
    font-size:12px;
    overflow:auto;
    max-height:300px
  }
  @media (max-width:1400px){
    .grid-kpi{
      grid-template-columns:repeat(5,1fr)
    }
    .main,.bottom{
      grid-template-columns:1fr
    }
  }
  @media (max-width:800px){
    .grid-kpi{
      grid-template-columns:repeat(2,1fr)
    }
    .io-grid,.controls,.charts-grid,.mini-grid{
      grid-template-columns:1fr
    }
  }
</style>
</head>
<body>
<div class="wrap">
  <div class="header">
    <div>
      <h1>PLC - Silo de Harina con 2 Silos + Mezcladora</h1>
      <div class="sub">Conmutación automática · transporte neumático · dosificación · ladder documentado · HMI/SCADA</div>
    </div>
    <div class="toolbar">
      <button class="b-green" onclick="startSim()">Iniciar</button>
      <button class="b-orange" onclick="pauseSim()">Pausar</button>
      <button class="b-blue" onclick="stepSim()">Paso</button>
      <button class="b-purple" onclick="resetSim()">Reset</button>
      <button class="b-red" onclick="emergencyStop()">Paro emergencia</button>
      <div class="chip"><label>Velocidad</label><input id="speedRange" type="range" min="200" max="1800" step="100" value="700"><span id="speedLabel">700 ms</span></div>
    </div>
  </div>

  <div class="grid-kpi">
    <div class="card kpi"><div class="label">Modo</div><div class="value blue" id="kpiModo">AUTO</div><div class="subv">HS-101</div></div>
    <div class="card kpi"><div class="label">Silo activo</div><div class="value purple" id="kpiSiloActivo">S1</div><div class="subv">Conmutación automática</div></div>
    <div class="card kpi"><div class="label">Nivel Silo 1</div><div class="value" id="kpiS1">0%</div><div class="subv">LT-101</div></div>
    <div class="card kpi"><div class="label">Nivel Silo 2</div><div class="value" id="kpiS2">0%</div><div class="subv">LT-201</div></div>
    <div class="card kpi"><div class="label">Tolva mezcladora</div><div class="value" id="kpiMixer">0%</div><div class="subv">LT-301</div></div>
    <div class="card kpi"><div class="label">Demanda proceso</div><div class="value" id="kpiDemanda">0%</div><div class="subv">MW10</div></div>
    <div class="card kpi"><div class="label">Entrada neumática</div><div class="value" id="kpiIn">0.0</div><div class="subv">t/h equiv.</div></div>
    <div class="card kpi"><div class="label">Descarga a mixer</div><div class="value" id="kpiOut">0.0</div><div class="subv">t/h equiv.</div></div>
    <div class="card kpi"><div class="label">Estado proceso</div><div class="value green" id="kpiEstado">Estable</div><div class="subv" id="kpiEstadoSub">Operación normal</div></div>
    <div class="card kpi"><div class="label">Alarmas activas</div><div class="value amber" id="kpiAlarmas">0</div><div class="subv">LL/L/H/HH/E-STOP</div></div>
  </div>

  <div class="main">
    <div class="card">
      <div class="panel-title">Sinótico de proceso</div>
      <div class="process-box">
        <svg id="plantSvg" viewBox="0 0 980 560" width="100%" height="560">
          <text x="30" y="40" fill="#dbeafe" font-size="18" font-weight="700">Recepción / transporte neumático</text>
          <rect id="blower" x="30" y="65" width="120" height="44" rx="12" fill="#475569" stroke="#93c5fd" stroke-width="2"/>
          <text x="50" y="92" fill="white" font-size="14">BL-101</text>
          <rect id="rotary" x="170" y="65" width="120" height="44" rx="12" fill="#475569" stroke="#93c5fd" stroke-width="2"/>
          <text x="184" y="92" fill="white" font-size="14">RV-101</text>
          <line id="pipeMain" x1="290" y1="87" x2="780" y2="87" stroke="#64748b" stroke-width="10" stroke-linecap="round"/>

          <!-- diverter -->
          <rect id="diverter" x="500" y="60" width="80" height="54" rx="12" fill="#334155" stroke="#93c5fd" stroke-width="2"/>
          <text x="511" y="92" fill="white" font-size="13">DV-101</text>
          <line id="branchS1" x1="540" y1="114" x2="540" y2="145" stroke="#64748b" stroke-width="8" />
          <line id="branchS2" x1="720" y1="114" x2="720" y2="145" stroke="#64748b" stroke-width="8" />

          <!-- Silo 1 -->
          <text x="440" y="165" fill="#dbeafe" font-size="18" font-weight="700">SILO 1</text>
          <path d="M390 185 L520 185 L545 330 L365 330 Z" fill="#0f172a" stroke="#cbd5e1" stroke-width="3"/>
          <path d="M365 330 L545 330 L455 425 Z" fill="#0f172a" stroke="#cbd5e1" stroke-width="3"/>
          <polygon id="fillS1" points="390,330 520,330 455,425" fill="#f5f5f4" opacity="0.92"></polygon>
          <rect x="425" y="160" width="60" height="24" rx="8" fill="#1d4ed8" stroke="#93c5fd" stroke-width="2"/><text x="436" y="176" fill="white" font-size="12">LT-101</text>

          <!-- Silo 2 -->
          <text x="620" y="165" fill="#dbeafe" font-size="18" font-weight="700">SILO 2</text>
          <path d="M570 185 L700 185 L725 330 L545 330 Z" fill="#0f172a" stroke="#cbd5e1" stroke-width="3"/>
          <path d="M545 330 L725 330 L635 425 Z" fill="#0f172a" stroke="#cbd5e1" stroke-width="3"/>
          <polygon id="fillS2" points="570,330 700,330 635,425" fill="#f5f5f4" opacity="0.92"></polygon>
          <rect x="605" y="160" width="60" height="24" rx="8" fill="#1d4ed8" stroke="#93c5fd" stroke-width="2"/><text x="616" y="176" fill="white" font-size="12">LT-201</text>

          <!-- outlet screws -->
          <rect id="cvS1" x="378" y="435" width="154" height="24" rx="10" fill="#475569" stroke="#93c5fd" stroke-width="2"/>
          <text x="403" y="451" fill="white" font-size="13">CV-102 S1</text>
          <rect id="cvS2" x="558" y="435" width="154" height="24" rx="10" fill="#475569" stroke="#93c5fd" stroke-width="2"/>
          <text x="583" y="451" fill="white" font-size="13">CV-202 S2</text>

          <!-- mixer -->
          <text x="790" y="220" fill="#dbeafe" font-size="18" font-weight="700">MEZCLADORA</text>
          <rect x="790" y="240" width="130" height="170" rx="18" fill="#0f172a" stroke="#cbd5e1" stroke-width="3"/>
          <rect id="fillMixer" x="805" y="370" width="100" height="25" rx="10" fill="#f5f5f4" opacity="0.92"/>
          <rect x="820" y="214" width="70" height="24" rx="8" fill="#1d4ed8" stroke="#93c5fd" stroke-width="2"/><text x="831" y="230" fill="white" font-size="12">LT-301</text>
          <rect id="agit" x="835" y="417" width="40" height="40" rx="8" fill="#475569" stroke="#93c5fd" stroke-width="2"/>
          <text x="828" y="444" fill="white" font-size="12">AG-301</text>

          <!-- pipes to mixer -->
          <line id="lineS1ToMix" x1="530" y1="447" x2="790" y2="447" stroke="#64748b" stroke-width="8" stroke-linecap="round"/>
          <line id="lineS2ToMix" x1="710" y1="447" x2="790" y2="447" stroke="#64748b" stroke-width="8" stroke-linecap="round"/>

          <!-- animated flow -->
          <circle id="p1" cx="320" cy="87" r="7" fill="#38bdf8" opacity="0"/>
          <circle id="p2" cx="360" cy="87" r="7" fill="#38bdf8" opacity="0"/>
          <circle id="p3" cx="400" cy="87" r="7" fill="#38bdf8" opacity="0"/>
          <circle id="o1" cx="560" cy="447" r="7" fill="#22c55e" opacity="0"/>
          <circle id="o2" cx="610" cy="447" r="7" fill="#22c55e" opacity="0"/>
          <circle id="o3" cx="660" cy="447" r="7" fill="#22c55e" opacity="0"/>

          <text x="30" y="530" fill="#94a3b8" font-size="12">
            Estrategia: llenar silo activo; si su nivel cae bajo LL se conmuta al otro silo disponible. Dosificación automática a mezcladora según nivel y demanda.
          </text>
        </svg>

        <div class="mini-grid">
          <div class="mini"><div class="label">LSLL S1</div><div class="value" id="mS1LL">0</div></div>
          <div class="mini"><div class="label">LSHH S1</div><div class="value" id="mS1HH">0</div></div>
          <div class="mini"><div class="label">LSLL S2</div><div class="value" id="mS2LL">0</div></div>
          <div class="mini"><div class="label">LSHH S2</div><div class="value" id="mS2HH">0</div></div>
        </div>
      </div>
    </div>

    <div class="right-stack">
      <div class="card">
        <div class="panel-title">Entradas / salidas PLC</div>
        <div class="io-grid">
          <div class="io"><div class="name">Modo</div><div class="state" id="ioMode">AUTO</div><div class="tag">HS-101</div></div>
          <div class="io"><div class="name">Paro emergencia</div><div class="state" id="ioEStop">OFF</div><div class="tag">ES-101</div></div>
          <div class="io"><div class="name">Soplador neumático</div><div class="state" id="ioBlower"><span class="lamp off"></span>OFF</div><div class="tag">Q0.0 / BL-101</div></div>
          <div class="io"><div class="name">Válvula rotativa</div><div class="state" id="ioRotary"><span class="lamp off"></span>OFF</div><div class="tag">Q0.1 / RV-101</div></div>
          <div class="io"><div class="name">Desviadora S1</div><div class="state" id="ioDivS1"><span class="lamp off"></span>OFF</div><div class="tag">Q0.2 / DV-S1</div></div>
          <div class="io"><div class="name">Desviadora S2</div><div class="state" id="ioDivS2"><span class="lamp off"></span>OFF</div><div class="tag">Q0.3 / DV-S2</div></div>
          <div class="io"><div class="name">Tornillo salida S1</div><div class="state" id="ioCv1"><span class="lamp off"></span>OFF</div><div class="tag">Q0.4 / CV-102</div></div>
          <div class="io"><div class="name">Tornillo salida S2</div><div class="state" id="ioCv2"><span class="lamp off"></span>OFF</div><div class="tag">Q0.5 / CV-202</div></div>
          <div class="io"><div class="name">Agitador mixer</div><div class="state" id="ioAgit"><span class="lamp off"></span>OFF</div><div class="tag">Q0.6 / AG-301</div></div>
          <div class="io"><div class="name">Nivel mixer</div><div class="state" id="ioMix">0%</div><div class="tag">IW90 / LT-301</div></div>
        </div>
      </div>

      <div class="card">
        <div class="panel-title">Controles HMI</div>
        <div class="controls">
          <button class="b-blue" onclick="toggleMode()">Cambiar AUTO/MAN</button>
          <button class="b-gray" onclick="ackAlarms()">Reconocer alarmas</button>
          <button class="b-green" onclick="manualToggle('blower')">Manual BL-101</button>
          <button class="b-green" onclick="manualToggle('rotary')">Manual RV-101</button>
          <button class="b-green" onclick="manualToggle('divS1')">Manual DV→S1</button>
          <button class="b-green" onclick="manualToggle('divS2')">Manual DV→S2</button>
          <button class="b-green" onclick="manualToggle('cv1')">Manual CV-102</button>
          <button class="b-green" onclick="manualToggle('cv2')">Manual CV-202</button>
          <button class="b-green" onclick="manualToggle('agit')">Manual AG-301</button>
          <button class="b-purple" onclick="setDemand('low')">Demanda baja</button>
          <button class="b-purple" onclick="setDemand('med')">Demanda media</button>
          <button class="b-purple" onclick="setDemand('high')">Demanda alta</button>
          <button class="b-red" onclick="forceAlarm('s1')">Prueba HH S1</button>
          <button class="b-red" onclick="forceAlarm('s2')">Prueba HH S2</button>
        </div>
        <div class="footer-note">En automático el PLC decide llenado, conmutación de silo fuente y dosificación a mezcladora.</div>
      </div>

      <div class="card">
        <div class="panel-title">Alarmas y eventos</div>
        <div class="alarms" id="alarmBox"></div>
      </div>
    </div>
  </div>

  <div class="bottom">
    <div class="card">
      <div class="panel-title">Tendencias</div>
      <div class="charts-grid">
        <div class="chart-box"><canvas id="chartLevel"></canvas></div>
        <div class="chart-box"><canvas id="chartMixer"></canvas></div>
      </div>
      <div class="footer-note">Tendencias históricas de ambos silos y tolva de mezcladora.</div>
    </div>

    <div class="card">
      <div class="panel-title">Tabla de tags PLC</div>
      <div class="table-wrap">
        <table>
          <thead><tr><th>Tag</th><th>Dirección</th><th>Descripción</th><th>Valor</th></tr></thead>
          <tbody id="tagTable"></tbody>
        </table>
      </div>
    </div>
  </div>

  <div style="height:16px"></div>
  <div class="bottom">
    <div class="card">
      <div class="panel-title">Lógica Ladder documentada</div>
<pre>
RUNG 1 - MODO AUTOMÁTICO
|----[ HS-101 AUTO ]---------------------------------------------( M_AUTO )----|

RUNG 2 - PARO DE EMERGENCIA
|----[ ES-101 ]--------------------------------------------------( ALM_ESTOP )---|
|----[ /ES-101 ]-------------------------------------------------( PERMISO )---|

RUNG 3 - LLENADO SILO ACTIVO
|----[ M_AUTO ]--[ PERMISO ]--[ NIVEL_SILO_ACTIVO < 75 ]---------( BL-101 )----|
|----[ M_AUTO ]--[ PERMISO ]--[ NIVEL_SILO_ACTIVO < 75 ]---------( RV-101 )----|

RUNG 4 - DESVIADORA DE LÍNEA
|----[ SILO_ACTIVO = S1 ]----------------------------------------( DV_S1 )------|
|----[ SILO_ACTIVO = S2 ]----------------------------------------( DV_S2 )------|

RUNG 5 - CONMUTACIÓN AUTOMÁTICA
|----[ SILO_ACTIVO = S1 ]--[ LT-101 <= 10 ]--[ LT-201 > 10 ]-----( CAMBIAR_S2 )|
|----[ SILO_ACTIVO = S2 ]--[ LT-201 <= 10 ]--[ LT-101 > 10 ]-----( CAMBIAR_S1 )|

RUNG 6 - DOSIFICACIÓN A MEZCLADORA
|----[ M_AUTO ]--[ LT-301 < 65 ]--[ SILO_ACTIVO = S1 ]----------( CV-102 )-----|
|----[ M_AUTO ]--[ LT-301 < 65 ]--[ SILO_ACTIVO = S2 ]----------( CV-202 )-----|
|----[ LT-301 < 80 ]--------------------------------------------( AG-301 )------|

RUNG 7 - ALARMAS
|----[ LT-101 >= 90 ]-------------------------------------------( ALM_HH_S1 )---|
|----[ LT-101 <= 10 ]-------------------------------------------( ALM_LL_S1 )---|
|----[ LT-201 >= 90 ]-------------------------------------------( ALM_HH_S2 )---|
|----[ LT-201 <= 10 ]-------------------------------------------( ALM_LL_S2 )---|
</pre>
    </div>

    <div class="card">
      <div class="panel-title">Secuencia funcional</div>
<pre>
1) El operador selecciona AUTO.
2) Si no hay emergencia, el PLC habilita transporte neumático.
3) La desviadora dirige el producto al silo activo.
4) El silo activo se llena hasta banda alta.
5) La mezcladora consume harina según la demanda del proceso.
6) Cuando la tolva de mezcladora baja, el PLC abre el tornillo del silo activo.
7) Si el silo activo cae a nivel muy bajo, conmuta al otro silo disponible.
8) Si ambos silos quedan bajos, mantiene alarmas y limita dosificación.
9) Si un silo llega a HH, detiene llenado de ese silo.
10) Con E-STOP, todas las salidas van a OFF.
</pre>
    </div>
  </div>
</div>

<script>
const CFG = {
  tickMs: 700,
  histMax: 80,
  lowLow:10, low:25, high:75, highHigh:90,
  inletBase:1.15,
  outletBase:0.70,
  mixerConsumeBase:0.55,
  noise:0.08
};

let timer = null, scan = 0;
let hist = {labels:[], s1:[], s2:[], mix:[]};
let alarmLog = [];

const plc = {
  auto:true, eStop:false, active:'S1', demand:55,
  s1:55, s2:82, mixer:44,
  out:{
    blower:false, rotary:false, divS1:true, divS2:false, cv1:false, cv2:false, agit:false
  },
  man:{blower:false, rotary:false, divS1:false, divS2:false, cv1:false, cv2:false, agit:false},
  al:{s1LL:false,s1L:false,s1H:false,s1HH:false,s2LL:false,s2L:false,s2H:false,s2HH:false, estop:false}
};

function clamp(v,min,max){ return Math.max(min, Math.min(max,v)); }
function rnd(a,b){ return Math.random()*(b-a)+a; }
function ts(){ return new Date().toLocaleTimeString(); }
function activeAlarmCount(){ return Object.values(plc.al).filter(Boolean).length; }

function pushEvent(cls,msg){
  alarmLog.unshift({cls,msg,time:ts()});
  alarmLog = alarmLog.slice(0,140);
  document.getElementById('alarmBox').innerHTML = alarmLog.map(a => `<div class="alarm-line alarm-${a.cls}">[${a.time}] ${a.msg}</div>`).join('');
}

function lamp(on, alarm=false){
  if(alarm) return '<span class="lamp al"></span>ON';
  return `<span class="lamp ${on?'on':'off'}"></span>${on?'ON':'OFF'}`;
}

function updateDiscrete(){
  const prev = {...plc.al};
  plc.al.s1LL = plc.s1 <= CFG.lowLow; plc.al.s1L = plc.s1 <= CFG.low; plc.al.s1H = plc.s1 >= CFG.high; plc.al.s1HH = plc.s1 >= CFG.highHigh;
  plc.al.s2LL = plc.s2 <= CFG.lowLow; plc.al.s2L = plc.s2 <= CFG.low; plc.al.s2H = plc.s2 >= CFG.high; plc.al.s2HH = plc.s2 >= CFG.highHigh;
  plc.al.estop = plc.eStop;

  if(plc.al.s1LL && !prev.s1LL) pushEvent('hh','ALM-LL S1: nivel muy bajo');
  if(plc.al.s1HH && !prev.s1HH) pushEvent('hh','ALM-HH S1: nivel muy alto');
  if(plc.al.s2LL && !prev.s2LL) pushEvent('hh','ALM-LL S2: nivel muy bajo');
  if(plc.al.s2HH && !prev.s2HH) pushEvent('hh','ALM-HH S2: nivel muy alto');
  if(plc.al.estop && !prev.estop) pushEvent('hh','ALM-E-STOP: emergencia activa');
}

function plcLogic(){
  if(plc.eStop){
    Object.keys(plc.out).forEach(k => plc.out[k] = false);
    return;
  }

  if(plc.auto){
    // conmutación automática
    if(plc.active === 'S1' && plc.s1 <= CFG.lowLow && plc.s2 > CFG.lowLow){
      plc.active = 'S2';
      pushEvent('ok','Conmutación automática: Silo activo = S2');
    } else if(plc.active === 'S2' && plc.s2 <= CFG.lowLow && plc.s1 > CFG.lowLow){
      plc.active = 'S1';
      pushEvent('ok','Conmutación automática: Silo activo = S1');
    }

    // transporte neumático hacia silo activo si no está alto
    const activeLvl = plc.active === 'S1' ? plc.s1 : plc.s2;
    const canFill = activeLvl < CFG.high && !((plc.active === 'S1' && plc.al.s1HH) || (plc.active === 'S2' && plc.al.s2HH));
    plc.out.blower = canFill;
    plc.out.rotary = canFill;
    plc.out.divS1 = plc.active === 'S1';
    plc.out.divS2 = plc.active === 'S2';

    // dosificación a mezcladora
    plc.out.cv1 = false; plc.out.cv2 = false;
    if(plc.mixer < 65){
      if(plc.active === 'S1' && plc.s1 > CFG.lowLow + 1) plc.out.cv1 = true;
      if(plc.active === 'S2' && plc.s2 > CFG.lowLow + 1) plc.out.cv2 = true;
    }
    plc.out.agit = plc.mixer < 80;
  } else {
    for (const k of Object.keys(plc.out)) plc.out[k] = plc.man[k];
  }
}

function processModel(){
  let inletRate = 0, outRate = 0;
  const d = plc.demand / 100;

  if(plc.out.blower && plc.out.rotary){
    inletRate = CFG.inletBase + rnd(-CFG.noise, CFG.noise);
    if(plc.out.divS1) plc.s1 = clamp(plc.s1 + inletRate, 0, 100);
    if(plc.out.divS2) plc.s2 = clamp(plc.s2 + inletRate, 0, 100);
  }

  const doseRate = (CFG.outletBase + 1.0*d) + rnd(-CFG.noise, CFG.noise);
  if(plc.out.cv1){
    outRate = doseRate;
    plc.s1 = clamp(plc.s1 - outRate, 0, 100);
    plc.mixer = clamp(plc.mixer + outRate*0.9, 0, 100);
  }
  if(plc.out.cv2){
    outRate = doseRate;
    plc.s2 = clamp(plc.s2 - outRate, 0, 100);
    plc.mixer = clamp(plc.mixer + outRate*0.9, 0, 100);
  }

  // consumo del mixer por amasado
  const mixerUse = (CFG.mixerConsumeBase + 1.1*d) + rnd(-CFG.noise, CFG.noise);
  plc.mixer = clamp(plc.mixer - mixerUse, 0, 100);

  return {inletRate:Math.max(0,inletRate), outRate:Math.max(0,outRate)};
}

function demandProfile(){
  const base = 52 + 28*Math.sin(scan/10);
  const pert = rnd(-8, 8);
  plc.demand = clamp(base + pert, 10, 100);
}

function health(){
  if(plc.eStop) return {txt:'Paro', cls:'red', sub:'Emergencia activa'};
  if(plc.al.s1HH || plc.al.s1LL || plc.al.s2HH || plc.al.s2LL) return {txt:'Alarma', cls:'red', sub:'Revisar niveles'};
  if(plc.out.blower) return {txt:'Llenando', cls:'green', sub:'Transporte neumático activo'};
  if(plc.out.cv1 || plc.out.cv2) return {txt:'Dosificando', cls:'blue', sub:'Alimentando mezcladora'};
  return {txt:'Estable', cls:'green', sub:'Operación normal'};
}

function updateSVG(){
  function setSilo(polyId, level, xLeft, xRight, xMid){
    let points;
    if(level <= 34.5){
      const k = (level / 34.5);
      const leftX = xMid - 90*k, rightX = xMid + 90*k;
      points = `${leftX},330 ${rightX},330 ${xMid},425`;
      if(k > .98) points = `${xLeft},330 ${xRight},330 ${xMid},425`;
    } else {
      const extra = Math.min(145, (level-34.5)*2.9);
      const yTop = 330 - extra;
      points = `${xLeft},${yTop} ${xRight},${yTop} ${xRight},330 ${xMid+90},330 ${xMid},425 ${xMid-90},330 ${xLeft},330`;
    }
    document.getElementById(polyId).setAttribute('points', points);
  }
  setSilo('fillS1', plc.s1, 390, 520, 455);
  setSilo('fillS2', plc.s2, 570, 700, 635);

  const mixH = 140 * (plc.mixer/100);
  document.getElementById('fillMixer').setAttribute('y', 395 - mixH);
  document.getElementById('fillMixer').setAttribute('height', Math.max(12,mixH));

  document.getElementById('blower').setAttribute('fill', plc.out.blower ? '#22c55e' : '#475569');
  document.getElementById('rotary').setAttribute('fill', plc.out.rotary ? '#22c55e' : '#475569');
  document.getElementById('diverter').setAttribute('fill', (plc.out.divS1||plc.out.divS2) ? '#22c55e' : '#334155');
  document.getElementById('branchS1').setAttribute('stroke', plc.out.divS1 ? '#38bdf8' : '#64748b');
  document.getElementById('branchS2').setAttribute('stroke', plc.out.divS2 ? '#38bdf8' : '#64748b');
  document.getElementById('cvS1').setAttribute('fill', plc.out.cv1 ? '#22c55e' : '#475569');
  document.getElementById('cvS2').setAttribute('fill', plc.out.cv2 ? '#22c55e' : '#475569');
  document.getElementById('agit').setAttribute('fill', plc.out.agit ? '#22c55e' : '#475569');
  document.getElementById('lineS1ToMix').setAttribute('stroke', plc.out.cv1 ? '#22c55e' : '#64748b');
  document.getElementById('lineS2ToMix').setAttribute('stroke', plc.out.cv2 ? '#22c55e' : '#64748b');
  document.getElementById('pipeMain').setAttribute('stroke', plc.out.blower ? '#38bdf8' : '#64748b');

  ['p1','p2','p3'].forEach((id,i)=>{
    const el = document.getElementById(id);
    el.setAttribute('opacity', plc.out.blower ? '1' : '0');
    el.setAttribute('cx', 320 + ((scan+i*2)%12)*25);
  });
  ['o1','o2','o3'].forEach((id,i)=>{
    const el = document.getElementById(id);
    const active = plc.out.cv1 || plc.out.cv2;
    el.setAttribute('opacity', active ? '1' : '0');
    el.setAttribute('cx', 560 + ((scan+i*2)%10)*25);
  });
}

function updateKPIs(r){
  document.getElementById('kpiModo').textContent = plc.auto ? 'AUTO' : 'MAN';
  document.getElementById('kpiModo').className = 'value ' + (plc.auto ? 'blue' : 'amber');
  document.getElementById('kpiSiloActivo').textContent = plc.active;
  document.getElementById('kpiS1').textContent = plc.s1.toFixed(1)+'%';
  document.getElementById('kpiS2').textContent = plc.s2.toFixed(1)+'%';
  document.getElementById('kpiMixer').textContent = plc.mixer.toFixed(1)+'%';
  document.getElementById('kpiDemanda').textContent = plc.demand.toFixed(0)+'%';
  document.getElementById('kpiIn').textContent = r.inletRate.toFixed(1);
  document.getElementById('kpiOut').textContent = r.outRate.toFixed(1);
  document.getElementById('kpiAlarmas').textContent = activeAlarmCount();

  const h = health();
  document.getElementById('kpiEstado').textContent = h.txt;
  document.getElementById('kpiEstado').className = 'value ' + h.cls;
  document.getElementById('kpiEstadoSub').textContent = h.sub;

  document.getElementById('ioMode').textContent = plc.auto ? 'AUTO' : 'MANUAL';
  document.getElementById('ioEStop').innerHTML = plc.eStop ? '<span class="lamp al"></span>ON' : '<span class="lamp off"></span>OFF';
  document.getElementById('ioBlower').innerHTML = lamp(plc.out.blower);
  document.getElementById('ioRotary').innerHTML = lamp(plc.out.rotary);
  document.getElementById('ioDivS1').innerHTML = lamp(plc.out.divS1);
  document.getElementById('ioDivS2').innerHTML = lamp(plc.out.divS2);
  document.getElementById('ioCv1').innerHTML = lamp(plc.out.cv1);
  document.getElementById('ioCv2').innerHTML = lamp(plc.out.cv2);
  document.getElementById('ioAgit').innerHTML = lamp(plc.out.agit);
  document.getElementById('ioMix').textContent = plc.mixer.toFixed(1)+'%';

  document.getElementById('mS1LL').textContent = plc.al.s1LL ? '1' : '0';
  document.getElementById('mS1HH').textContent = plc.al.s1HH ? '1' : '0';
  document.getElementById('mS2LL').textContent = plc.al.s2LL ? '1' : '0';
  document.getElementById('mS2HH').textContent = plc.al.s2HH ? '1' : '0';

  updateSVG();
  renderTags(r);
}

function renderTags(r){
  const rows = [
    ['LT-101','IW64','Nivel silo 1', plc.s1.toFixed(1)+' %'],
    ['LT-201','IW74','Nivel silo 2', plc.s2.toFixed(1)+' %'],
    ['LT-301','IW90','Nivel mezcladora', plc.mixer.toFixed(1)+' %'],
    ['HS-101','I0.0','Modo automático', plc.auto ? '1':'0'],
    ['ES-101','I0.1','Paro emergencia', plc.eStop ? '1':'0'],
    ['BL-101','Q0.0','Soplador neumático', plc.out.blower ? '1':'0'],
    ['RV-101','Q0.1','Válvula rotativa', plc.out.rotary ? '1':'0'],
    ['DV-S1','Q0.2','Desviadora a silo 1', plc.out.divS1 ? '1':'0'],
    ['DV-S2','Q0.3','Desviadora a silo 2', plc.out.divS2 ? '1':'0'],
    ['CV-102','Q0.4','Tornillo salida silo 1', plc.out.cv1 ? '1':'0'],
    ['CV-202','Q0.5','Tornillo salida silo 2', plc.out.cv2 ? '1':'0'],
    ['AG-301','Q0.6','Agitador mezcladora', plc.out.agit ? '1':'0'],
    ['MW10','MW10','Demanda proceso', plc.demand.toFixed(0)+' %'],
    ['MW20','MW20','Caudal entrada', r.inletRate.toFixed(2)],
    ['MW21','MW21','Caudal dosificación', r.outRate.toFixed(2)],
    ['MW30','MW30','Silo activo', plc.active],
    ['MW31','MW31','Scan PLC', scan]
  ];
  document.getElementById('tagTable').innerHTML = rows.map(t => `<tr><td>${t[0]}</td><td>${t[1]}</td><td>${t[2]}</td><td>${t[3]}</td></tr>`).join('');
}

let chartLevel = null, chartMixer = null;
function initCharts(){
  const commonOpts = {
    responsive:true, maintainAspectRatio:false,
    plugins:{legend:{labels:{color:'#eef4ff'}}},
    scales:{
      x:{ticks:{color:'#cbd5e1'},grid:{color:'rgba(255,255,255,.06)'}},
      y:{min:0,max:100,ticks:{color:'#cbd5e1'},grid:{color:'rgba(255,255,255,.06)'}}
    }
  };
  chartLevel = new Chart(document.getElementById('chartLevel'), {
    type:'line',
    data:{labels:[], datasets:[
      {label:'Silo 1', data:[], borderColor:'#38bdf8', backgroundColor:'rgba(56,189,248,.15)', fill:false, tension:.3},
      {label:'Silo 2', data:[], borderColor:'#8b5cf6', backgroundColor:'rgba(139,92,246,.15)', fill:false, tension:.3}
    ]},
    options: commonOpts
  });
  chartMixer = new Chart(document.getElementById('chartMixer'), {
    type:'line',
    data:{labels:[], datasets:[
      {label:'Mezcladora', data:[], borderColor:'#22c55e', backgroundColor:'rgba(34,197,94,.15)', fill:true, tension:.3}
    ]},
    options: commonOpts
  });
}

function refreshCharts(){
  chartLevel.data.labels = hist.labels;
  chartLevel.data.datasets[0].data = hist.s1;
  chartLevel.data.datasets[1].data = hist.s2;
  chartLevel.update();

  chartMixer.data.labels = hist.labels;
  chartMixer.data.datasets[0].data = hist.mix;
  chartMixer.update();
}

function tick(){
  scan++;
  if(plc.auto && !plc.eStop) demandProfile();
  plcLogic();
  const r = processModel();
  updateDiscrete();

  hist.labels.push(scan);
  hist.s1.push(Number(plc.s1.toFixed(1)));
  hist.s2.push(Number(plc.s2.toFixed(1)));
  hist.mix.push(Number(plc.mixer.toFixed(1)));
  if(hist.labels.length > CFG.histMax){
    hist.labels.shift(); hist.s1.shift(); hist.s2.shift(); hist.mix.shift();
  }

  updateKPIs(r);
  refreshCharts();
}

function startSim(){ if(timer) return; pushEvent('i','Simulación iniciada'); timer = setInterval(tick, CFG.tickMs); }
function pauseSim(){ if(timer){ clearInterval(timer); timer = null; pushEvent('i','Simulación pausada'); } }
function stepSim(){ pauseSim(); tick(); pushEvent('i','Paso manual ejecutado'); }
function emergencyStop(){
  plc.eStop = !plc.eStop;
  pushEvent(plc.eStop ? 'hh' : 'ok', plc.eStop ? 'Paro de emergencia activado' : 'Paro de emergencia liberado');
  tick();
}
function toggleMode(){
  plc.auto = !plc.auto;
  pushEvent('i','Modo cambiado a ' + (plc.auto ? 'AUTO' : 'MANUAL'));
  updateKPIs({inletRate:0,outRate:0});
}
function manualToggle(key){
  if(plc.auto){ pushEvent('h','Ignorado: pasar a MANUAL'); return; }
  plc.man[key] = !plc.man[key];
  pushEvent('i',`Salida manual ${key} = ${plc.man[key] ? 'ON':'OFF'}`);
  tick();
}
function ackAlarms(){ pushEvent('ok','Alarmas reconocidas por operador'); }
function setDemand(kind){
  const m = {low:20, med:55, high:90};
  plc.demand = m[kind];
  pushEvent('i','Demanda fijada en ' + plc.demand + '%');
  updateKPIs({inletRate:0,outRate:0});
}
function forceAlarm(silo){
  if(silo === 's1') plc.s1 = 93;
  if(silo === 's2') plc.s2 = 93;
  pushEvent('hh','Prueba HH forzada en ' + silo.toUpperCase());
  tick();
}
function resetSim(){
  pauseSim();
  scan = 0;
  hist = {labels:[], s1:[], s2:[], mix:[]};
  alarmLog = [];
  plc.auto = true; plc.eStop = false; plc.active = 'S1'; plc.demand = 55;
  plc.s1 = 55; plc.s2 = 82; plc.mixer = 44;
  plc.out = {blower:false, rotary:false, divS1:true, divS2:false, cv1:false, cv2:false, agit:false};
  plc.man = {blower:false, rotary:false, divS1:false, divS2:false, cv1:false, cv2:false, agit:false};
  plc.al = {s1LL:false,s1L:false,s1H:false,s1HH:false,s2LL:false,s2L:false,s2H:false,s2HH:false, estop:false};
  document.getElementById('alarmBox').innerHTML = '';
  pushEvent('ok','Sistema reiniciado');
  updateKPIs({inletRate:0,outRate:0});
  refreshCharts();
}

document.getElementById('speedRange').addEventListener('input', e => {
  CFG.tickMs = Number(e.target.value);
  document.getElementById('speedLabel').textContent = CFG.tickMs + ' ms';
  if(timer){ clearInterval(timer); timer = setInterval(tick, CFG.tickMs); }
});

initCharts();
resetSim();
</script>
</body>
</html>
"""

# Find the script block containing the simulation logic
script_match = re.search(r'<script>(.*?)</script>', html, re.DOTALL)

if script_match:
    js_code = script_match.group(1).strip()
    print('Extracted JavaScript simulation code:')
    print(js_code)
else:
    print('No JavaScript script block found in the HTML.')

Extracted JavaScript simulation code:
const CFG = {
  tickMs: 700,
  histMax: 80,
  lowLow:10, low:25, high:75, highHigh:90,
  inletBase:1.15,
  outletBase:0.70,
  mixerConsumeBase:0.55,
  noise:0.08
};

let timer = null, scan = 0;
let hist = {labels:[], s1:[], s2:[], mix:[]};
let alarmLog = [];

const plc = {
  auto:true, eStop:false, active:'S1', demand:55,
  s1:55, s2:82, mixer:44,
  out:{
    blower:false, rotary:false, divS1:true, divS2:false, cv1:false, cv2:false, agit:false
  },
  man:{blower:false, rotary:false, divS1:false, divS2:false, cv1:false, cv2:false, agit:false},
  al:{s1LL:false,s1L:false,s1H:false,s1HH:false,s2LL:false,s2L:false,s2H:false,s2HH:false, estop:false}
};

function clamp(v,min,max){ return Math.max(min, Math.min(max,v)); }
function rnd(a,b){ return Math.random()*(b-a)+a; }
function ts(){ return new Date().toLocaleTimeString(); }
function activeAlarmCount(){ return Object.values(plc.al).filter(Boolean).length; }

function pushEvent(cls,msg){
  alarmLog.unsh

In [6]:
%run /content/plc_silos_harina_colab.py

Exception: File `'/content/plc_silos_harina_colab.py'` not found.

In [7]:

# ============================================
# Simulación EEG para análisis del sueño
# Rol: Ingeniero Biomédico
# Diseñado para ejecutarse en Google Colab
# ============================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------
# Configuración general
# -----------------------------
FS = 128                    # frecuencia de muestreo [Hz]
EPOCH_SEC = 30              # duración por época [s]
N_EPOCHS = 20               # cantidad de épocas simuladas
SEED = 42                   # reproducibilidad

rng = np.random.default_rng(SEED)

# Secuencia típica simplificada de sueño
SLEEP_SEQUENCE = [
    "Awake", "N1", "N2", "N2", "N3",
    "N3", "N2", "REM", "N2", "N3",
    "N2", "REM", "N2", "N1", "N2",
    "N3", "REM", "N2", "REM", "Awake"
]

# -----------------------------
# Modelos sintéticos por etapa
# -----------------------------
def simulate_stage_signal(stage: str, duration_s: int, fs: int, rng: np.random.Generator):
    t = np.arange(0, duration_s, 1/fs)
    n = len(t)

    # ruido basal
    noise = rng.normal(0, 5, n)

    if stage == "Awake":
        # alfa + beta, amplitud moderada
        sig = (
            20*np.sin(2*np.pi*10*t) +   # alpha
            8*np.sin(2*np.pi*18*t) +    # beta
            noise
        )
    elif stage == "N1":
        # theta
        sig = (
            25*np.sin(2*np.pi*6*t) +
            6*np.sin(2*np.pi*3*t) +
            noise
        )
    elif stage == "N2":
        # theta + husos de sueño + complejos K
        sig = 20*np.sin(2*np.pi*5*t) + noise

        # husos: ráfagas 12-14 Hz
        for _ in range(3):
            center = rng.uniform(3, duration_s-3)
            spindle = np.exp(-0.5*((t-center)/0.5)**2) * 18*np.sin(2*np.pi*13*t)
            sig += spindle

        # complejos K: pulsos lentos
        for _ in range(2):
            center = rng.uniform(4, duration_s-4)
            k_complex = -40*np.exp(-((t-center)/0.15)**2) + 25*np.exp(-((t-(center+0.2))/0.25)**2)
            sig += k_complex
    elif stage == "N3":
        # delta predominante, alta amplitud
        sig = (
            60*np.sin(2*np.pi*1.5*t) +
            25*np.sin(2*np.pi*0.8*t) +
            noise
        )
    elif stage == "REM":
        # baja amplitud, frecuencia mixta similar a vigilia
        sig = (
            12*np.sin(2*np.pi*7*t) +
            10*np.sin(2*np.pi*16*t) +
            noise
        )
    else:
        sig = noise

    # deriva lenta de base
    baseline = 5*np.sin(2*np.pi*0.15*t)
    sig = sig + baseline
    return t, sig


def stage_band_powers(stage: str, rng: np.random.Generator):
    """
    Valores sintéticos relativos para bandas:
    delta, theta, alpha, beta
    """
    if stage == "Awake":
        vals = [0.10, 0.20, 0.40, 0.30]
    elif stage == "N1":
        vals = [0.15, 0.50, 0.20, 0.15]
    elif stage == "N2":
        vals = [0.25, 0.40, 0.15, 0.20]
    elif stage == "N3":
        vals = [0.70, 0.20, 0.05, 0.05]
    elif stage == "REM":
        vals = [0.10, 0.35, 0.15, 0.40]
    else:
        vals = [0.25, 0.25, 0.25, 0.25]

    vals = np.array(vals) + rng.normal(0, 0.02, 4)
    vals = np.clip(vals, 0.01, None)
    vals = vals / vals.sum()
    return vals


def build_full_signal(sequence, epoch_sec=30, fs=128, rng=None):
    t_all = []
    x_all = []
    stage_per_sample = []
    epoch_summary = []

    current_time = 0.0
    for i, stage in enumerate(sequence):
        t, x = simulate_stage_signal(stage, epoch_sec, fs, rng)
        t_shifted = t + current_time

        t_all.append(t_shifted)
        x_all.append(x)
        stage_per_sample.extend([stage] * len(t))

        band_vals = stage_band_powers(stage, rng)
        epoch_summary.append({
            "epoch": i + 1,
            "stage": stage,
            "start_s": current_time,
            "end_s": current_time + epoch_sec,
            "delta": band_vals[0],
            "theta": band_vals[1],
            "alpha": band_vals[2],
            "beta": band_vals[3],
            "rms_uV": float(np.sqrt(np.mean(x**2))),
            "peak_uV": float(np.max(np.abs(x)))
        })

        current_time += epoch_sec

    return (
        np.concatenate(t_all),
        np.concatenate(x_all),
        np.array(stage_per_sample),
        pd.DataFrame(epoch_summary)
    )


def create_sleep_dashboard(t, x, summary_df, fs=128):
    stage_numeric = {
        "Awake": 4,
        "REM": 3,
        "N1": 2,
        "N2": 1,
        "N3": 0
    }

    hypnogram_y = [stage_numeric[s] for s in summary_df["stage"]]
    stage_labels = ["N3", "N2", "N1", "REM", "Awake"]

    # ventana para mostrar EEG detalle
    detail_sec = 60
    idx = t <= detail_sec
    t_detail = t[idx]
    x_detail = x[idx]

    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.07,
        row_heights=[0.35, 0.20, 0.22, 0.23],
        subplot_titles=(
            "EEG simulado (primeros 60 s)",
            "Hipnograma",
            "Potencia relativa por bandas",
            "Indicadores por época"
        )
    )

    # 1) EEG
    fig.add_trace(
        go.Scatter(
            x=t_detail,
            y=x_detail,
            mode="lines",
            name="EEG"
        ),
        row=1, col=1
    )

    # 2) Hipnograma
    fig.add_trace(
        go.Scatter(
            x=summary_df["start_s"] / 60,
            y=hypnogram_y,
            mode="lines+markers",
            line_shape="hv",
            name="Etapa"
        ),
        row=2, col=1
    )

    # 3) Potencia relativa por bandas
    epochs = summary_df["epoch"]
    fig.add_trace(go.Bar(x=epochs, y=summary_df["delta"], name="Delta"), row=3, col=1)
    fig.add_trace(go.Bar(x=epochs, y=summary_df["theta"], name="Theta"), row=3, col=1)
    fig.add_trace(go.Bar(x=epochs, y=summary_df["alpha"], name="Alpha"), row=3, col=1)
    fig.add_trace(go.Bar(x=epochs, y=summary_df["beta"], name="Beta"), row=3, col=1)

    # 4) RMS y pico
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=summary_df["rms_uV"],
            mode="lines+markers",
            name="RMS (uV)"
        ),
        row=4, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=summary_df["peak_uV"],
            mode="lines+markers",
            name="Pico (uV)"
        ),
        row=4, col=1
    )

    fig.update_yaxes(title_text="uV", row=1, col=1)
    fig.update_xaxes(title_text="Tiempo [s]", row=1, col=1)

    fig.update_yaxes(
        title_text="Etapa",
        tickmode="array",
        tickvals=[0,1,2,3,4],
        ticktext=stage_labels,
        row=2, col=1
    )
    fig.update_xaxes(title_text="Tiempo [min]", row=2, col=1)

    fig.update_yaxes(title_text="Potencia relativa", row=3, col=1)
    fig.update_xaxes(title_text="Época", row=3, col=1)

    fig.update_yaxes(title_text="uV", row=4, col=1)
    fig.update_xaxes(title_text="Época", row=4, col=1)

    fig.update_layout(
        height=1100,
        title="Dashboard profesional - Simulación EEG para análisis del sueño",
        barmode="stack",
        template="plotly_dark",
        legend_title="Variables"
    )
    return fig


def create_summary_cards(summary_df):
    total_min = (summary_df["end_s"].max() - summary_df["start_s"].min()) / 60
    stage_counts = summary_df["stage"].value_counts()

    sleep_min = total_min - stage_counts.get("Awake", 0) * EPOCH_SEC / 60
    rem_min = stage_counts.get("REM", 0) * EPOCH_SEC / 60
    n3_min = stage_counts.get("N3", 0) * EPOCH_SEC / 60
    efficiency = 100 * sleep_min / total_min if total_min > 0 else 0

    cards = pd.DataFrame({
        "Indicador": [
            "Tiempo total registrado [min]",
            "Tiempo total de sueño [min]",
            "Sueño REM [min]",
            "Sueño profundo N3 [min]",
            "Eficiencia de sueño [%]",
            "RMS EEG promedio [uV]"
        ],
        "Valor": [
            round(total_min, 1),
            round(sleep_min, 1),
            round(rem_min, 1),
            round(n3_min, 1),
            round(efficiency, 1),
            round(summary_df["rms_uV"].mean(), 1)
        ]
    })
    return cards


# -----------------------------
# Ejecutar simulación
# -----------------------------
t, x, stage_per_sample, summary_df = build_full_signal(
    SLEEP_SEQUENCE,
    epoch_sec=EPOCH_SEC,
    fs=FS,
    rng=rng
)

cards_df = create_summary_cards(summary_df)
fig = create_sleep_dashboard(t, x, summary_df, fs=FS)

print("Resumen clínico-técnico sintético")
print(cards_df.to_string(index=False))

print("\nResumen por época")
print(summary_df[["epoch", "stage", "delta", "theta", "alpha", "beta", "rms_uV", "peak_uV"]].round(3).to_string(index=False))

fig.show()

# Exportación opcional
summary_df.to_csv("sleep_eeg_summary.csv", index=False)
cards_df.to_csv("sleep_eeg_kpis.csv", index=False)

print("\nArchivos generados:")
print("- sleep_eeg_summary.csv")
print("- sleep_eeg_kpis.csv")


Resumen clínico-técnico sintético
                    Indicador  Valor
Tiempo total registrado [min]   10.0
  Tiempo total de sueño [min]    9.0
              Sueño REM [min]    2.0
      Sueño profundo N3 [min]    2.0
      Eficiencia de sueño [%]   90.0
        RMS EEG promedio [uV]   21.9

Resumen por época
 epoch stage  delta  theta  alpha  beta  rms_uV  peak_uV
     1 Awake  0.071  0.196  0.401 0.333  16.378   43.314
     2    N1  0.139  0.484  0.211 0.165  19.079   43.202
     3    N2  0.253  0.370  0.182 0.195  16.457   48.822
     4    N2  0.243  0.378  0.165 0.213  16.243   62.200
     5    N3  0.723  0.230  0.027 0.020  46.345   96.987
     6    N3  0.692  0.212  0.043 0.054  46.331   98.939
     7    N2  0.271  0.378  0.179 0.172  16.325   49.709
     8   REM  0.133  0.354  0.136 0.376  12.665   35.862
     9    N2  0.235  0.411  0.172 0.182  16.259   56.356
    10    N3  0.724  0.215  0.044 0.017  46.368   98.374
    11    N2  0.243  0.400  0.162 0.195  16.219   51.850
    


Archivos generados:
- sleep_eeg_summary.csv
- sleep_eeg_kpis.csv


In [15]:
from textwrap import dedent
from pathlib import Path

code = dedent(r'''
# ============================================
# Simulación EEG para análisis del sueño
# Rol: Ingeniero Biomédico
# Diseñado para ejecutarse en Google Colab
# ============================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------
# Configuración general
# -----------------------------
FS = 128                    # frecuencia de muestreo [Hz]
EPOCH_SEC = 30              # duración por época [s]
N_EPOCHS = 20               # cantidad de épocas simuladas
SEED = 42                   # reproducibilidad

rng = np.random.default_rng(SEED)

# Secuencia típica simplificada de sueño
SLEEP_SEQUENCE = [
    "Awake", "N1", "N2", "N2", "N3",
    "N3", "N2", "REM", "N2", "N3",
    "N2", "REM", "N2", "N1", "N2",
    "N3", "REM", "N2", "REM", "Awake"
]

# -----------------------------
# Modelos sintéticos por etapa
# -----------------------------
def simulate_stage_signal(stage: str, duration_s: int, fs: int, rng: np.random.Generator):
    t = np.arange(0, duration_s, 1/fs)
    n = len(t)

    # ruido basal
    noise = rng.normal(0, 5, n)

    if stage == "Awake":
        # alfa + beta, amplitud moderada
        sig = (
            20*np.sin(2*np.pi*10*t) +   # alpha
            8*np.sin(2*np.pi*18*t) +    # beta
            noise
        )
    elif stage == "N1":
        # theta
        sig = (
            25*np.sin(2*np.pi*6*t) +
            6*np.sin(2*np.pi*3*t) +
            noise
        )
    elif stage == "N2":
        # theta + husos de sueño + complejos K
        sig = 20*np.sin(2*np.pi*5*t) + noise

        # husos: ráfagas 12-14 Hz
        for _ in range(3):
            center = rng.uniform(3, duration_s-3)
            spindle = np.exp(-0.5*((t-center)/0.5)**2) * 18*np.sin(2*np.pi*13*t)
            sig += spindle

        # complejos K: pulsos lentos
        for _ in range(2):
            center = rng.uniform(4, duration_s-4)
            k_complex = -40*np.exp(-((t-center)/0.15)**2) + 25*np.exp(-((t-(center+0.2))/0.25)**2)
            sig += k_complex
    elif stage == "N3":
        # delta predominante, alta amplitud
        sig = (
            60*np.sin(2*np.pi*1.5*t) +
            25*np.sin(2*np.pi*0.8*t) +
            noise
        )
    elif stage == "REM":
        # baja amplitud, frecuencia mixta similar a vigilia
        sig = (
            12*np.sin(2*np.pi*7*t) +
            10*np.sin(2*np.pi*16*t) +
            noise
        )
    else:
        sig = noise

    # deriva lenta de base
    baseline = 5*np.sin(2*np.pi*0.15*t)
    sig = sig + baseline
    return t, sig


def stage_band_powers(stage: str, rng: np.random.Generator):
    """
    Valores sintéticos relativos para bandas:
    delta, theta, alpha, beta
    """
    if stage == "Awake":
        vals = [0.10, 0.20, 0.40, 0.30]
    elif stage == "N1":
        vals = [0.15, 0.50, 0.20, 0.15]
    elif stage == "N2":
        vals = [0.25, 0.40, 0.15, 0.20]
    elif stage == "N3":
        vals = [0.70, 0.20, 0.05, 0.05]
    elif stage == "REM":
        vals = [0.10, 0.35, 0.15, 0.40]
    else:
        vals = [0.25, 0.25, 0.25, 0.25]

    vals = np.array(vals) + rng.normal(0, 0.02, 4)
    vals = np.clip(vals, 0.01, None)
    vals = vals / vals.sum()
    return vals


def build_full_signal(sequence, epoch_sec=30, fs=128, rng=None):
    t_all = []
    x_all = []
    stage_per_sample = []
    epoch_summary = []

    current_time = 0.0
    for i, stage in enumerate(sequence):
        t, x = simulate_stage_signal(stage, epoch_sec, fs, rng)
        t_shifted = t + current_time

        t_all.append(t_shifted)
        x_all.append(x)
        stage_per_sample.extend([stage] * len(t))

        band_vals = stage_band_powers(stage, rng)
        epoch_summary.append({
            "epoch": i + 1,
            "stage": stage,
            "start_s": current_time,
            "end_s": current_time + epoch_sec,
            "delta": band_vals[0],
            "theta": band_vals[1],
            "alpha": band_vals[2],
            "beta": band_vals[3],
            "rms_uV": float(np.sqrt(np.mean(x**2))),
            "peak_uV": float(np.max(np.abs(x)))
        })

        current_time += epoch_sec

    return (
        np.concatenate(t_all),
        np.concatenate(x_all),
        np.array(stage_per_sample),
        pd.DataFrame(epoch_summary)
    )


def create_sleep_dashboard(t, x, summary_df, fs=128):
    stage_numeric = {
        "Awake": 4,
        "REM": 3,
        "N1": 2,
        "N2": 1,
        "N3": 0
    }

    hypnogram_y = [stage_numeric[s] for s in summary_df["stage"]]
    stage_labels = ["N3", "N2", "N1", "REM", "Awake"]

    # ventana para mostrar EEG detalle
    detail_sec = 60
    idx = t <= detail_sec
    t_detail = t[idx]
    x_detail = x[idx]

    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.07,
        row_heights=[0.35, 0.20, 0.22, 0.23],
        subplot_titles=(
            "EEG simulado (primeros 60 s)",
            "Hipnograma",
            "Potencia relativa por bandas",
            "Indicadores por época"
        )
    )

    # 1) EEG
    fig.add_trace(
        go.Scatter(
            x=t_detail,
            y=x_detail,
            mode="lines",
            name="EEG"
        ),
        row=1, col=1
    )

    # 2) Hipnograma
    fig.add_trace(
        go.Scatter(
            x=summary_df["start_s"] / 60,
            y=hypnogram_y,
            mode="lines+markers",
            line_shape="hv",
            name="Etapa"
        ),
        row=2, col=1
    )

    # 3) Potencia relativa por bandas
    epochs = summary_df["epoch"]
    fig.add_trace(go.Bar(x=epochs, y=summary_df["delta"], name="Delta"), row=3, col=1)
    fig.add_trace(go.Bar(x=epochs, y=summary_df["theta"], name="Theta"), row=3, col=1)
    fig.add_trace(go.Bar(x=epochs, y=summary_df["alpha"], name="Alpha"), row=3, col=1)
    fig.add_trace(go.Bar(x=epochs, y=summary_df["beta"], name="Beta"), row=3, col=1)

    # 4) RMS y pico
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=summary_df["rms_uV"],
            mode="lines+markers",
            name="RMS (uV)"
        ),
        row=4, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=summary_df["peak_uV"],
            mode="lines+markers",
            name="Pico (uV)"
        ),
        row=4, col=1
    )

    fig.update_yaxes(title_text="uV", row=1, col=1)
    fig.update_xaxes(title_text="Tiempo [s]", row=1, col=1)

    fig.update_yaxes(
        title_text="Etapa",
        tickmode="array",
        tickvals=[0,1,2,3,4],
        ticktext=stage_labels,
        row=2, col=1
    )
    fig.update_xaxes(title_text="Tiempo [min]", row=2, col=1)

    fig.update_yaxes(title_text="Potencia relativa", row=3, col=1)
    fig.update_xaxes(title_text="Época", row=3, col=1)

    fig.update_yaxes(title_text="uV", row=4, col=1)
    fig.update_xaxes(title_text="Época", row=4, col=1)

    fig.update_layout(
        height=1100,
        title="Dashboard profesional - Simulación EEG para análisis del sueño",
        barmode="stack",
        template="plotly_dark",
        legend_title="Variables"
    )
    return fig


def create_summary_cards(summary_df):
    total_min = (summary_df["end_s"].max() - summary_df["start_s"].min()) / 60
    stage_counts = summary_df["stage"].value_counts()

    sleep_min = total_min - stage_counts.get("Awake", 0) * EPOCH_SEC / 60
    rem_min = stage_counts.get("REM", 0) * EPOCH_SEC / 60
    n3_min = stage_counts.get("N3", 0) * EPOCH_SEC / 60
    efficiency = 100 * sleep_min / total_min if total_min > 0 else 0

    cards = pd.DataFrame({
        "Indicador": [
            "Tiempo total registrado [min]",
            "Tiempo total de sueño [min]",
            "Sueño REM [min]",
            "Sueño profundo N3 [min]",
            "Eficiencia de sueño [%]",
            "RMS EEG promedio [uV]"
        ],
        "Valor": [
            round(total_min, 1),
            round(sleep_min, 1),
            round(rem_min, 1),
            round(n3_min, 1),
            round(efficiency, 1),
            round(summary_df["rms_uV"].mean(), 1)
        ]
    })
    return cards


# -----------------------------
# Ejecutar simulación
# -----------------------------
t, x, stage_per_sample, summary_df = build_full_signal(
    SLEEP_SEQUENCE,
    epoch_sec=EPOCH_SEC,
    fs=FS,
    rng=rng
)

cards_df = create_summary_cards(summary_df)
fig = create_sleep_dashboard(t, x, summary_df, fs=FS)

print("Resumen clínico-técnico sintético")
print(cards_df.to_string(index=False))

print("\nResumen por época")
print(summary_df[["epoch", "stage", "delta", "theta", "alpha", "beta", "rms_uV", "peak_uV"]].round(3).to_string(index=False))

fig.show()

# Exportación opcional
summary_df.to_csv("sleep_eeg_summary.csv", index=False)
cards_df.to_csv("sleep_eeg_kpis.csv", index=False)

print("\nArchivos generados:")
print("- sleep_eeg_summary.csv")
print("- sleep_eeg_kpis.csv")
''')

path = Path("/content/eeg_sueno_dashboard_colab.py")
path.write_text(code, encoding="utf-8")
print(path)

/content/eeg_sueno_dashboard_colab.py


In [14]:
import pandas as pd

sleep_summary_df = pd.read_csv('sleep_eeg_summary.csv')
display(sleep_summary_df.head())

,epoch,stage,start_s,end_s,delta,theta,alpha,beta,rms_uV,peak_uV
0,1,Awake,0.0,30.0,0.071380,0.195598,0.400507,0.332515,16.377569,43.313632
1,2,N1,30.0,60.0,0.139307,0.484317,0.211297,0.165079,19.079207,43.201837
2,3,N2,60.0,90.0,0.253156,0.369818,0.181599,0.195428,16.456835,48.821556
3,4,N2,90.0,120.0,0.243398,0.378114,0.165017,0.213471,16.243289,62.200332
4,5,N3,120.0,150.0,0.723147,0.230058,0.027115,0.019680,46.344836,96.987475


In [17]:

# ============================================
# Simulación EEG para análisis del sueño
# Rol: Ingeniero Biomédico
# Diseñado para ejecutarse en Google Colab
# ============================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------
# Configuración general
# -----------------------------
FS = 128                    # frecuencia de muestreo [Hz]
EPOCH_SEC = 30              # duración por época [s]
N_EPOCHS = 20               # cantidad de épocas simuladas
SEED = 42                   # reproducibilidad

rng = np.random.default_rng(SEED)

# Secuencia típica simplificada de sueño
SLEEP_SEQUENCE = [
    "Awake", "N1", "N2", "N2", "N3",
    "N3", "N2", "REM", "N2", "N3",
    "N2", "REM", "N2", "N1", "N2",
    "N3", "REM", "N2", "REM", "Awake"
]

# -----------------------------
# Modelos sintéticos por etapa
# -----------------------------
def simulate_stage_signal(stage: str, duration_s: int, fs: int, rng: np.random.Generator):
    t = np.arange(0, duration_s, 1/fs)
    n = len(t)

    # ruido basal
    noise = rng.normal(0, 5, n)

    if stage == "Awake":
        # alfa + beta, amplitud moderada
        sig = (
            20*np.sin(2*np.pi*10*t) +   # alpha
            8*np.sin(2*np.pi*18*t) +    # beta
            noise
        )
    elif stage == "N1":
        # theta
        sig = (
            25*np.sin(2*np.pi*6*t) +
            6*np.sin(2*np.pi*3*t) +
            noise
        )
    elif stage == "N2":
        # theta + husos de sueño + complejos K
        sig = 20*np.sin(2*np.pi*5*t) + noise

        # husos: ráfagas 12-14 Hz
        for _ in range(3):
            center = rng.uniform(3, duration_s-3)
            spindle = np.exp(-0.5*((t-center)/0.5)**2) * 18*np.sin(2*np.pi*13*t)
            sig += spindle

        # complejos K: pulsos lentos
        for _ in range(2):
            center = rng.uniform(4, duration_s-4)
            k_complex = -40*np.exp(-((t-center)/0.15)**2) + 25*np.exp(-((t-(center+0.2))/0.25)**2)
            sig += k_complex
    elif stage == "N3":
        # delta predominante, alta amplitud
        sig = (
            60*np.sin(2*np.pi*1.5*t) +
            25*np.sin(2*np.pi*0.8*t) +
            noise
        )
    elif stage == "REM":
        # baja amplitud, frecuencia mixta similar a vigilia
        sig = (
            12*np.sin(2*np.pi*7*t) +
            10*np.sin(2*np.pi*16*t) +
            noise
        )
    else:
        sig = noise

    # deriva lenta de base
    baseline = 5*np.sin(2*np.pi*0.15*t)
    sig = sig + baseline
    return t, sig


def stage_band_powers(stage: str, rng: np.random.Generator):
    """
    Valores sintéticos relativos para bandas:
    delta, theta, alpha, beta
    """
    if stage == "Awake":
        vals = [0.10, 0.20, 0.40, 0.30]
    elif stage == "N1":
        vals = [0.15, 0.50, 0.20, 0.15]
    elif stage == "N2":
        vals = [0.25, 0.40, 0.15, 0.20]
    elif stage == "N3":
        vals = [0.70, 0.20, 0.05, 0.05]
    elif stage == "REM":
        vals = [0.10, 0.35, 0.15, 0.40]
    else:
        vals = [0.25, 0.25, 0.25, 0.25]

    vals = np.array(vals) + rng.normal(0, 0.02, 4)
    vals = np.clip(vals, 0.01, None)
    vals = vals / vals.sum()
    return vals


def build_full_signal(sequence, epoch_sec=30, fs=128, rng=None):
    t_all = []
    x_all = []
    stage_per_sample = []
    epoch_summary = []

    current_time = 0.0
    for i, stage in enumerate(sequence):
        t, x = simulate_stage_signal(stage, epoch_sec, fs, rng)
        t_shifted = t + current_time

        t_all.append(t_shifted)
        x_all.append(x)
        stage_per_sample.extend([stage] * len(t))

        band_vals = stage_band_powers(stage, rng)
        epoch_summary.append({
            "epoch": i + 1,
            "stage": stage,
            "start_s": current_time,
            "end_s": current_time + epoch_sec,
            "delta": band_vals[0],
            "theta": band_vals[1],
            "alpha": band_vals[2],
            "beta": band_vals[3],
            "rms_uV": float(np.sqrt(np.mean(x**2))),
            "peak_uV": float(np.max(np.abs(x)))
        })

        current_time += epoch_sec

    return (
        np.concatenate(t_all),
        np.concatenate(x_all),
        np.array(stage_per_sample),
        pd.DataFrame(epoch_summary)
    )


def create_sleep_dashboard(t, x, summary_df, fs=128):
    stage_numeric = {
        "Awake": 4,
        "REM": 3,
        "N1": 2,
        "N2": 1,
        "N3": 0
    }

    hypnogram_y = [stage_numeric[s] for s in summary_df["stage"]]
    stage_labels = ["N3", "N2", "N1", "REM", "Awake"]

    # ventana para mostrar EEG detalle
    detail_sec = 60
    idx = t <= detail_sec
    t_detail = t[idx]
    x_detail = x[idx]

    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.07,
        row_heights=[0.35, 0.20, 0.22, 0.23],
        subplot_titles=(
            "EEG simulado (primeros 60 s)",
            "Hipnograma",
            "Potencia relativa por bandas",
            "Indicadores por época"
        )
    )

    # 1) EEG
    fig.add_trace(
        go.Scatter(
            x=t_detail,
            y=x_detail,
            mode="lines",
            name="EEG"
        ),
        row=1, col=1
    )

    # 2) Hipnograma
    fig.add_trace(
        go.Scatter(
            x=summary_df["start_s"] / 60,
            y=hypnogram_y,
            mode="lines+markers",
            line_shape="hv",
            name="Etapa"
        ),
        row=2, col=1
    )

    # 3) Potencia relativa por bandas
    epochs = summary_df["epoch"]
    fig.add_trace(go.Bar(x=epochs, y=summary_df["delta"], name="Delta"), row=3, col=1)
    fig.add_trace(go.Bar(x=epochs, y=summary_df["theta"], name="Theta"), row=3, col=1)
    fig.add_trace(go.Bar(x=epochs, y=summary_df["alpha"], name="Alpha"), row=3, col=1)
    fig.add_trace(go.Bar(x=epochs, y=summary_df["beta"], name="Beta"), row=3, col=1)

    # 4) RMS y pico
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=summary_df["rms_uV"],
            mode="lines+markers",
            name="RMS (uV)"
        ),
        row=4, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=summary_df["peak_uV"],
            mode="lines+markers",
            name="Pico (uV)"
        ),
        row=4, col=1
    )

    fig.update_yaxes(title_text="uV", row=1, col=1)
    fig.update_xaxes(title_text="Tiempo [s]", row=1, col=1)

    fig.update_yaxes(
        title_text="Etapa",
        tickmode="array",
        tickvals=[0,1,2,3,4],
        ticktext=stage_labels,
        row=2, col=1
    )
    fig.update_xaxes(title_text="Tiempo [min]", row=2, col=1)

    fig.update_yaxes(title_text="Potencia relativa", row=3, col=1)
    fig.update_xaxes(title_text="Época", row=3, col=1)

    fig.update_yaxes(title_text="uV", row=4, col=1)
    fig.update_xaxes(title_text="Época", row=4, col=1)

    fig.update_layout(
        height=1100,
        title="Dashboard - Simulación EEG para análisis del sueño",
        barmode="stack",
        template="plotly_dark",
        legend_title="Variables"
    )
    return fig


def create_summary_cards(summary_df):
    total_min = (summary_df["end_s"].max() - summary_df["start_s"].min()) / 60
    stage_counts = summary_df["stage"].value_counts()

    sleep_min = total_min - stage_counts.get("Awake", 0) * EPOCH_SEC / 60
    rem_min = stage_counts.get("REM", 0) * EPOCH_SEC / 60
    n3_min = stage_counts.get("N3", 0) * EPOCH_SEC / 60
    efficiency = 100 * sleep_min / total_min if total_min > 0 else 0

    cards = pd.DataFrame({
        "Indicador": [
            "Tiempo total registrado [min]",
            "Tiempo total de sueño [min]",
            "Sueño REM [min]",
            "Sueño profundo N3 [min]",
            "Eficiencia de sueño [%]",
            "RMS EEG promedio [uV]"
        ],
        "Valor": [
            round(total_min, 1),
            round(sleep_min, 1),
            round(rem_min, 1),
            round(n3_min, 1),
            round(efficiency, 1),
            round(summary_df["rms_uV"].mean(), 1)
        ]
    })
    return cards


# -----------------------------
# Ejecutar simulación
# -----------------------------
t, x, stage_per_sample, summary_df = build_full_signal(
    SLEEP_SEQUENCE,
    epoch_sec=EPOCH_SEC,
    fs=FS,
    rng=rng
)

cards_df = create_summary_cards(summary_df)
fig = create_sleep_dashboard(t, x, summary_df, fs=FS)

print("Resumen clínico-técnico sintético")
print(cards_df.to_string(index=False))

print("\nResumen por época")
print(summary_df[["epoch", "stage", "delta", "theta", "alpha", "beta", "rms_uV", "peak_uV"]].round(3).to_string(index=False))

fig.show()

# Exportación opcional
summary_df.to_csv("sleep_eeg_summary.csv", index=False)
cards_df.to_csv("sleep_eeg_kpis.csv", index=False)

print("\nArchivos generados:")
print("- sleep_eeg_summary.csv")
print("- sleep_eeg_kpis.csv")


Resumen clínico-técnico sintético
                    Indicador  Valor
Tiempo total registrado [min]   10.0
  Tiempo total de sueño [min]    9.0
              Sueño REM [min]    2.0
      Sueño profundo N3 [min]    2.0
      Eficiencia de sueño [%]   90.0
        RMS EEG promedio [uV]   21.9

Resumen por época
 epoch stage  delta  theta  alpha  beta  rms_uV  peak_uV
     1 Awake  0.071  0.196  0.401 0.333  16.378   43.314
     2    N1  0.139  0.484  0.211 0.165  19.079   43.202
     3    N2  0.253  0.370  0.182 0.195  16.457   48.822
     4    N2  0.243  0.378  0.165 0.213  16.243   62.200
     5    N3  0.723  0.230  0.027 0.020  46.345   96.987
     6    N3  0.692  0.212  0.043 0.054  46.331   98.939
     7    N2  0.271  0.378  0.179 0.172  16.325   49.709
     8   REM  0.133  0.354  0.136 0.376  12.665   35.862
     9    N2  0.235  0.411  0.172 0.182  16.259   56.356
    10    N3  0.724  0.215  0.044 0.017  46.368   98.374
    11    N2  0.243  0.400  0.162 0.195  16.219   51.850
    


Archivos generados:
- sleep_eeg_summary.csv
- sleep_eeg_kpis.csv
